<div dir="rtl">

# מערכת לניהול סוכנות טיולי שטח
**קורס סימולציה, עבודת בית 2027A**

**חברי הקבוצה:** רועי גרינפלד, ניצן שמואלי, דניאל דיל, שגיא שניר

המחברת בונה מערכת מונחית עצמים בפייתון לניהול סוכנות טיולי שטח: טיולים, מטיילים, מדריכים, הרשמות, רשימות המתנה, מדיניות ביטולים ולוח זמנים עונתי. בנוסף, כשיפור יצירתי, יש שתי מחלקות חדשות: **ציוד** ו**הסעה**.

**איך מריצים:** בתפריט Runtime בוחרים Run all. הקוד משתמש רק בספרייה הסטנדרטית של פייתון.

**מבנה המחברת:**
1. הנחות וכללי עיצוב
2. המחלקות: `Trek`, `Traveler`, `Guide`, `Agency`, ואחריהן מחלקות ההרחבה `Equipment` ו-`Transport`
3. הדגמה מהירה
4. בדיקות והדגמות לכל מחלקה ומתודה: שימוש תקין, מקרי קצה, קלט לא תקין ופעולות אסורות
5. הסימולציה המלאה: תרחיש עונה ותרחישי קצה

</div>

<div dir="rtl">

## 1. הנחות וכללי עיצוב

**כללי עיצוב**
- במחלקות שהעבודה מגדירה יש רק השדות שהיא מגדירה. הוספנו מתודות ופרמטרים אופציונליים בלבד. שדות חדשים קיימים רק במחלקות ההרחבה.
- כל מתודה שמשנה מצב עובדת באותו סדר: ולידציה, בדיקת אילוצים, עדכון ורענון מצב נגזר. כך כישלון לא משאיר עדכון חלקי.
- קלט לא תקין או הפרת כלל זורקים `ValueError` לפני שמשהו משתנה. הקוד הקורא תופס את השגיאה עם `try/except`.
- בדיקת טיפוס: `type(x) is int` לערכים פשוטים, כדי לדחות `True`, ו-`isinstance` לאובייקטים.
- קשר דו-כיווני: מתודה אחת מבצעת את העדכון בשני הצדדים, והשנייה רק קוראת לה. למשל `Traveler.register_to_trek` קוראת ל-`Trek.register`.
- חפיפת תאריכים מחושבת במקום אחד בלבד, `Trek.overlaps`. יום הסיום נספר.
- דוחות מחושבים בכל קריאה ולא נשמרים.

**הנחות** (במקומות שהעבודה משאירה פתוחים)

| נושא | הבחירה שלנו |
| --- | --- |
| התאריך של היום | תמיד `date.today()` האמיתי. בהדגמות מזיזים את תאריכי הטיולים, לא את היום |
| תאריכי עבר | מותר ליצור טיול בתאריך עבר, ואסור להירשם לטיול שכבר יצא |
| החזר בביטול, לפי ימים לפני היציאה | 30 ומעלה: 100%. 14 עד 29: 75%. 7 עד 13: 50%. פחות מכך: 0% |
| הנחת מנוי | basic: 0%, annual: 10%, premium: 20% |
| סדר רשימת ההמתנה | premium, אחריו annual, אחריו basic, ובתוך כל סוג לפי סדר ההגעה |
| סולמות | דרגת קושי, ניסיון והסמכה: 1 עד 10 |
| גיל | מספר שלם בין 1 ל-120 |
| מגבלה חודשית של מדריך | נספרת לפי חודש היציאה |
| טיול פרטי | 500 ליום, באזור ההתמחות הראשון של המדריך, ברמת הניסיון של המטייל |
| עונות | לפי חודש היציאה: חורף דצמבר-פברואר, אביב מרץ-מאי, קיץ יוני-אוגוסט, סתיו ספטמבר-נובמבר |
| התרעות בלוח העונתי | טיולים פעילים באותו אזור שחולקים יום |
| הורשה | אין מחלקת בסיס `Person`. ל-`Traveler` ול-`Guide` כמעט אין התנהגות משותפת מעבר לשם ולבדיקת זמינות, ולכן העדפנו מחלקות נפרדות ופשוטות. ההיררכיה במערכת היא של הרכבה: סוכנות מחזיקה מטיילים, מדריכים וטיולים, וטיול מחזיק מדריך ומשתתפים |

**ייבוא:** כל הקוד נמצא במחברת אחת, ולכן נשארו רק ייבואים מהספרייה הסטנדרטית. ההגדרה של `Trek` לא תלויה במחלקות האחרות: היא מזהה מטייל ומדריך לפי שמות השדות שלהם, וכך אין ייבוא מעגלי.

</div>

In [1]:
import math
from datetime import date, datetime, timedelta
from pprint import pprint

<div dir="rtl">

## 2. המחלקות

### 2.1 `Trek`, טיול
המחלקה המרכזית במערכת. היא בודקת תאריכים וקיבולת, רושמת מטיילים, מנהלת את רשימת ההמתנה, מחשבת החזרים ומחזיקה את הסטטוס (`planned`, `confirmed`, `cancelled`, `completed`).
- `register` בודקת קיבולת, התאמת רמה והתנגשויות בלוח הזמנים של המטייל.
- `update_status` מאשרת טיול כשהוא מגיע למינימום המשתתפים, ומחזירה אותו ל"מתוכנן" כשהוא יורד מתחתיו.
- `promote_from_waiting_list` מקדמת ממתין לפי סוג המנוי.
- **תוספות:** המתודה הסטטית `overlaps`, שהיא המקום היחיד שבו מחושבת חפיפת תאריכים, והפרמטרים האופציונליים `cancel_date` ו-`completion_date`, שברירת המחדל שלהם היא היום.

</div>

In [2]:
class Trek:
    # Assumption: the agency operates only in Israel, in these regions. To add a region, add it here
    VALID_REGIONS=("Golan Heights", "Galilee", "Upper Galilee", "Lower Galilee", "Carmel",
                   "Judean Mountains", "Judean Desert", "Dead Sea", "Negev",
                   "Arava", "Eilat Mountains")
    # The only statuses a trek can be in
    VALID_STATUSES=("planned", "confirmed", "cancelled", "completed")

    # Creates a valid trek with no guide and no participants; an invalid value raises ValueError before any field is set
    def __init__(self, trek_id, name, region, difficulty_level, min_participants,
                 max_participants, start_date, duration_days, price):
        # Assumption: trek_id and difficulty_level are whole numbers
        if type(trek_id) is not int or trek_id<=0:
            raise ValueError(f"Invalid trek id: {trek_id}")
        if type(name) is not str or not name.strip():
            raise ValueError(f"Invalid name: {name}")
        if region not in Trek.VALID_REGIONS:
            raise ValueError(f"Invalid region: {region}")
        if type(difficulty_level) is not int or not 1<=difficulty_level<=10:
            raise ValueError(f"Invalid difficulty level: {difficulty_level}")
        if type(min_participants) is not int or type(max_participants) is not int:
            raise ValueError("Participant limits must be whole numbers")
        if min_participants<=0 or min_participants>max_participants:
            raise ValueError(f"Invalid participant limits: {min_participants}, {max_participants}")
        Trek.check_dates(start_date, duration_days)
        if type(price) not in (int, float) or not math.isfinite(price) or price<=0:
            raise ValueError(f"Invalid price: {price}")

        # Unique identifier of the trek
        self.trek_id=trek_id
        # Display name of the trek
        self.name=name
        # Region of the trek, one of VALID_REGIONS
        self.region=region
        # Difficulty on a scale of 1 to 10
        self.difficulty_level=difficulty_level
        # Minimum number of participants needed for the trek to depart
        self.min_participants=min_participants
        # Maximum number of participants the trek can hold
        self.max_participants=max_participants
        # Departure date, changed later only through set_dates
        self.start_date=start_date
        # Length of the trek in days, changed later only through set_dates
        self.duration_days=duration_days
        # Full price before discounts
        self.price=price
        # Assigned guide, None until assign_guide checks availability
        self.guide=None
        # Registered travelers, changed only through register, cancel_registration and cancel_trek
        self.participants=[]
        # Waiting travelers in order of arrival, changed only through add_to_waiting_list, promote_from_waiting_list and cancel_trek
        self.waiting_list=[]
        # Current status, one of VALID_STATUSES, updated by update_status, cancel_trek and complete_trek
        self.status="planned"

    # Validates a date range in one place; static, so Traveler and Guide can use it without a trek object
    @staticmethod
    def check_dates(start_date, duration_days):
        # type() rejects datetime, because comparing it with a date raises TypeError
        if type(start_date) is not date:
            raise ValueError(f"Invalid start date: {start_date}")
        if type(duration_days) is not int or duration_days<1:
            raise ValueError(f"Invalid duration: {duration_days}")

    # Rejects anything without a traveler id; checks the shared name rather than the class, so the trek file needs no circular import
    def check_traveler(self, traveler):
        if not hasattr(traveler, "traveler_id"):
            raise ValueError(f"Invalid traveler: {traveler}")

    # Rejects anything without a guide id, for the same reason as check_traveler
    def check_guide(self, guide):
        if not hasattr(guide, "guide_id"):
            raise ValueError(f"Invalid guide: {guide}")

    # Returns True while the trek can still change, meaning it is neither cancelled nor completed
    def is_active(self):
        return self.status in ("planned", "confirmed")

    # Returns the last day of the trek; the end day counts, so a one day trek ends on its start date
    def get_end_date(self):
        return self.start_date+timedelta(days=self.duration_days-1)

    # Returns True if the given date range shares at least one day with this trek
    def overlaps(self, start_date, duration_days):
        Trek.check_dates(start_date, duration_days)
        end_date=start_date+timedelta(days=duration_days-1)
        return start_date<=self.get_end_date() and self.start_date<=end_date

    # Returns True if the guide has no room left in the month of the given date, not counting this trek
    def is_guide_month_full(self, guide, start_date):
        count=0
        for trek in guide.assigned_treks:
            if trek is not self and trek.start_date.year==start_date.year and trek.start_date.month==start_date.month:
                count+=1
        return count>=guide.max_monthly_treks

    # Assigns a guide and updates both sides, after checking status, region, schedule and monthly limit
    def assign_guide(self, guide):
        self.check_guide(guide)
        if not self.is_active():
            raise ValueError(f"Cannot assign a guide to a {self.status} trek")
        if self.guide is not None:
            raise ValueError(f"Trek {self.trek_id} already has a guide")
        if self.region not in guide.specialties:
            raise ValueError(f"Guide {guide.guide_id} does not specialize in {self.region}")
        if not guide.is_available(self.start_date, self.duration_days):
            raise ValueError(f"Guide {guide.guide_id} is not available on these dates")
        if self.is_guide_month_full(guide, self.start_date):
            raise ValueError(f"Guide {guide.guide_id} reached the monthly trek limit")

        self.guide=guide
        guide.assigned_treks.append(self)

    # Rejects anything that is not a vehicle assigned to this trek, checking shared names for the same reason as check_traveler
    def check_vehicle(self, vehicle):
        if not hasattr(vehicle, "transport_id"):
            raise ValueError(f"Invalid vehicle: {vehicle}")
        if not vehicle.is_assigned_to(self.trek_id):
            raise ValueError(f"Vehicle {vehicle.transport_id} is not assigned to trek {self.trek_id}")

    # Rejects anything that is not equipment on loan to this trek, checking shared names for the same reason as check_traveler
    def check_equipment(self, item):
        if not hasattr(item, "equipment_id"):
            raise ValueError(f"Invalid equipment: {item}")
        if not item.is_loaned_to(self.trek_id):
            raise ValueError(f"Equipment {item.equipment_id} is not on loan to trek {self.trek_id}")

    # Sets new dates only if the guide, every participant and the given vehicles and equipment can make them, otherwise the old dates stay; a trek has no field for those two, so the calling code passes them
    def set_dates(self, start_date, duration_days, vehicles=None, equipment=None):
        Trek.check_dates(start_date, duration_days)
        if vehicles is None:
            vehicles=[]
        if equipment is None:
            equipment=[]
        if type(vehicles) is not list:
            raise ValueError(f"Invalid vehicles: {vehicles}")
        if type(equipment) is not list:
            raise ValueError(f"Invalid equipment: {equipment}")
        for vehicle in vehicles:
            self.check_vehicle(vehicle)
        for item in equipment:
            self.check_equipment(item)
        if not self.is_active():
            raise ValueError(f"Cannot change the dates of a {self.status} trek")
        # The trek's own id is ignored, otherwise it would conflict with itself
        if self.guide is not None:
            if not self.guide.is_available(start_date, duration_days, self.trek_id):
                raise ValueError(f"Guide {self.guide.guide_id} is not available on these dates")
            if self.is_guide_month_full(self.guide, start_date):
                raise ValueError(f"Guide {self.guide.guide_id} reached the monthly trek limit")
        for traveler in self.participants:
            if not traveler.is_available(start_date, duration_days, self.trek_id):
                raise ValueError(f"Traveler {traveler.traveler_id} is not available on these dates")
        for vehicle in vehicles:
            if not vehicle.is_available(start_date, duration_days, self.trek_id):
                raise ValueError(f"Vehicle {vehicle.transport_id} is not available on these dates")
        for item in equipment:
            if not item.can_supply(self.trek_id, start_date, duration_days):
                raise ValueError(f"Equipment {item.equipment_id} is not available on these dates")

        self.start_date=start_date
        self.duration_days=duration_days

    # Returns True if at least one place is left
    def has_capacity(self):
        return len(self.participants)<self.max_participants

    # Returns True if the traveler's experience reaches the trek's difficulty (assumption: both use the same scale)
    def is_suitable(self, traveler):
        self.check_traveler(traveler)
        return traveler.experience_level>=self.difficulty_level

    # Returns True once the departure day has arrived
    def has_departed(self):
        return self.start_date<=date.today()

    # Registers a traveler and updates both sides, only after every check has passed
    def register(self, traveler):
        self.check_traveler(traveler)
        if not self.is_active():
            raise ValueError(f"Cannot register to a {self.status} trek")
        if self.has_departed():
            raise ValueError(f"Trek {self.trek_id} has already departed")
        if traveler in self.participants:
            raise ValueError(f"Traveler {traveler.traveler_id} is already registered")
        if traveler in self.waiting_list:
            raise ValueError(f"Traveler {traveler.traveler_id} is on the waiting list")
        if not self.is_suitable(traveler):
            raise ValueError(f"Traveler {traveler.traveler_id} is not suitable for this trek")
        if not self.has_capacity():
            raise ValueError(f"Trek {self.trek_id} is full")
        if not traveler.is_available(self.start_date, self.duration_days):
            raise ValueError(f"Traveler {traveler.traveler_id} has a schedule conflict")

        self.participants.append(traveler)
        traveler.current_treks.append(self)
        self.update_status()

    # Cancels a registration on both sides, then promotes a waiting traveler and refreshes the status
    def cancel_registration(self, traveler_id):
        if not self.is_active():
            raise ValueError(f"Cannot cancel a registration of a {self.status} trek")
        for traveler in self.participants:
            if traveler.traveler_id==traveler_id:
                self.participants.remove(traveler)
                traveler.current_treks.remove(self)
                self.promote_from_waiting_list()
                self.update_status()
                return
        raise ValueError(f"Traveler {traveler_id} is not registered to trek {self.trek_id}")

    # Returns the refund rate between 0 and 1 by days before departure (assumption): 30+ days 100%, 14-29 days 75%, 7-13 days 50%, fewer 0%
    def calculate_refund_rate(self, cancel_date=None):
        # With no date the rate uses today; the optional parameter keeps results repeatable in tests
        if cancel_date is None:
            cancel_date=date.today()
        if type(cancel_date) is not date:
            raise ValueError(f"Invalid cancel date: {cancel_date}")
        days_before=(self.start_date-cancel_date).days
        if days_before>=30:
            return 1.0
        if days_before>=14:
            return 0.75
        if days_before>=7:
            return 0.5
        return 0.0

    # Adds a traveler to the end of the waiting list, only when the trek is full
    def add_to_waiting_list(self, traveler):
        self.check_traveler(traveler)
        if not self.is_active():
            raise ValueError(f"Cannot join the waiting list of a {self.status} trek")
        if self.has_departed():
            raise ValueError(f"Trek {self.trek_id} has already departed")
        if self.has_capacity():
            raise ValueError(f"Trek {self.trek_id} still has room, register instead")
        if traveler in self.participants:
            raise ValueError(f"Traveler {traveler.traveler_id} is already registered")
        if traveler in self.waiting_list:
            raise ValueError(f"Traveler {traveler.traveler_id} is already on the waiting list")
        if not self.is_suitable(traveler):
            raise ValueError(f"Traveler {traveler.traveler_id} is not suitable for this trek")

        self.waiting_list.append(traveler)

    # Promotes a waiting traveler by membership type, then by order of arrival within the type; anyone no longer available is skipped, and doing nothing is not an error
    def promote_from_waiting_list(self):
        if not self.has_capacity() or self.has_departed():
            return
        # Assumption: premium comes before annual, and annual before basic
        for membership_type in ("premium", "annual", "basic"):
            for traveler in self.waiting_list:
                if traveler.membership_type==membership_type and traveler.is_available(self.start_date, self.duration_days):
                    self.waiting_list.remove(traveler)
                    self.register(traveler)
                    return

    # Sets confirmed or planned by the number of participants; a cancelled or completed trek never changes back
    def update_status(self):
        if not self.is_active():
            return
        if len(self.participants)>=self.min_participants:
            self.status="confirmed"
        else:
            self.status="planned"

    # Cancels the trek on the agency's initiative and releases everyone (assumption: travelers owe nothing, so the trek just leaves their current treks)
    def cancel_trek(self):
        if not self.is_active():
            raise ValueError(f"Cannot cancel a {self.status} trek")
        for traveler in self.participants:
            traveler.current_treks.remove(self)
        self.participants=[]
        self.waiting_list=[]
        if self.guide is not None:
            self.guide.remove_trek(self.trek_id)
        self.status="cancelled"

    # Closes a confirmed trek that has ended and credits every participant; the optional date keeps tests repeatable
    def complete_trek(self, completion_date=None):
        if completion_date is None:
            completion_date=date.today()
        if type(completion_date) is not date:
            raise ValueError(f"Invalid completion date: {completion_date}")
        if self.status!="confirmed":
            raise ValueError(f"Cannot complete a {self.status} trek")
        if completion_date<self.get_end_date():
            raise ValueError(f"Trek {self.trek_id} has not ended yet")

        # The status changes first, because add_completed_trek accepts only a completed trek
        self.status="completed"
        for traveler in self.participants:
            traveler.add_completed_trek(self.trek_id)

<div dir="rtl">

### 2.2 `Traveler`, מטייל
מחזיק את הטיולים הנוכחיים ואת אלה שהושלמו, מבקש טיול פרטי ומפיק דוח תשלומים.
- `register_to_trek` ו-`cancel_trek` רק קוראות למתודות של `Trek`, כדי ששני הצדדים יתעדכנו יחד.
- `request_private_tour` יוצרת טיול חדש עם מקום אחד, משבצת אליו את המדריך ורושמת את המטייל, רק אם שניהם פנויים.
- **תוספת:** `get_final_price` מחשבת את המחיר אחרי הנחת המנוי. גם דוח התשלומים וגם דוח המנויים משתמשים בה.

</div>

In [3]:
class Traveler:
    # The only membership types a traveler can have
    VALID_MEMBERSHIPS=("basic", "annual", "premium")
    # Assumption: the discount rate of each membership type
    MEMBERSHIP_DISCOUNTS={"basic": 0.0, "annual": 0.1, "premium": 0.2}
    # Assumption: a private tour costs a fixed amount per day, because request_private_tour receives no price
    PRIVATE_TOUR_DAILY_PRICE=500

    # Creates a valid traveler with no treks; an invalid value raises ValueError before any field is set
    def __init__(self, traveler_id, name, age, membership_type, experience_level):
        if type(traveler_id) is not int or traveler_id<=0:
            raise ValueError(f"Invalid traveler id: {traveler_id}")
        if type(name) is not str or not name.strip():
            raise ValueError(f"Invalid name: {name}")
        # Assumption: 120 is the highest realistic age
        if type(age) is not int or not 1<=age<=120:
            raise ValueError(f"Invalid age: {age}")
        if membership_type not in Traveler.VALID_MEMBERSHIPS:
            raise ValueError(f"Invalid membership type: {membership_type}")
        # Assumption: the same 1 to 10 scale as a trek's difficulty level
        if type(experience_level) is not int or not 1<=experience_level<=10:
            raise ValueError(f"Invalid experience level: {experience_level}")

        # Unique identifier of the traveler
        self.traveler_id=traveler_id
        # Full name of the traveler
        self.name=name
        # Age of the traveler in years
        self.age=age
        # Membership type, one of VALID_MEMBERSHIPS
        self.membership_type=membership_type
        # Experience on the same scale as a trek's difficulty level
        self.experience_level=experience_level
        # Treks the traveler is registered to, changed only through Trek's methods and add_completed_trek
        self.current_treks=[]
        # Treks the traveler finished, changed only through add_completed_trek
        self.completed_treks=[]

    # Asks to register to a trek; Trek makes all the checks, so there is one place that registers
    def register_to_trek(self, trek):
        if not isinstance(trek, Trek):
            raise ValueError(f"Invalid trek: {trek}")
        trek.register(self)

    # Cancels a registration and returns the refund rate, which is calculated before the trek is released
    def cancel_trek(self, trek_id):
        for trek in self.current_treks:
            if trek.trek_id==trek_id:
                refund_rate=trek.calculate_refund_rate()
                trek.cancel_registration(self.traveler_id)
                return refund_rate
        raise ValueError(f"Traveler {self.traveler_id} is not registered to trek {trek_id}")

    # Moves a trek from current to completed, only after the trek itself was closed
    def add_completed_trek(self, trek_id):
        for trek in self.current_treks:
            if trek.trek_id==trek_id:
                if trek.status!="completed":
                    raise ValueError(f"Trek {trek_id} is not completed")
                self.current_treks.remove(trek)
                self.completed_treks.append(trek)
                return
        raise ValueError(f"Trek {trek_id} is not a current trek of traveler {self.traveler_id}")

    # Creates a one person trek with this guide and returns it; the calling code adds it to the agency
    def request_private_tour(self, guide, start_date, duration_days):
        # Checks the shared name rather than the class, because the guide file imports nothing from this one
        if not hasattr(guide, "guide_id"):
            raise ValueError(f"Invalid guide: {guide}")
        if not self.is_available(start_date, duration_days):
            raise ValueError(f"Traveler {self.traveler_id} is not available on these dates")

        # Assumption: an id built from the traveler and the date needs no counter, and a traveler cannot start two treks on one day
        trek_id=self.traveler_id*1000000+start_date.toordinal()
        # Assumption: the region is the guide's first specialty and the difficulty is the traveler's own level
        trek=Trek(trek_id, f"Private tour of {self.name}", guide.specialties[0], self.experience_level,
                  1, 1, start_date, duration_days, Traveler.PRIVATE_TOUR_DAILY_PRICE*duration_days)
        # Checked here because the guide is assigned before the registration that would reject this date
        if trek.has_departed():
            raise ValueError("Cannot request a private tour that has already departed")

        trek.assign_guide(guide)
        trek.register(self)
        return trek

    # Returns True if the date range overlaps none of the current treks; ignore_trek_id lets a trek skip itself when it moves
    def is_available(self, start_date, duration_days, ignore_trek_id=None):
        Trek.check_dates(start_date, duration_days)
        for trek in self.current_treks:
            if trek.trek_id!=ignore_trek_id and trek.overlaps(start_date, duration_days):
                return False
        return True

    # Returns the trek's price after the membership discount, without changing the trek's full price
    def get_final_price(self, trek):
        if not isinstance(trek, Trek):
            raise ValueError(f"Invalid trek: {trek}")
        return round(trek.price*(1-Traveler.MEMBERSHIP_DISCOUNTS[self.membership_type]), 2)

    # Returns a payment report calculated on every call; a current trek also shows its fee if cancelled today, a past cancellation has no field to be kept in
    def generate_payment_report(self):
        treks=[]
        total=0
        for trek in self.current_treks:
            final_price=self.get_final_price(trek)
            cancellation_fee=round(final_price*(1-trek.calculate_refund_rate()), 2)
            treks.append({"trek_id": trek.trek_id, "name": trek.name, "state": "current",
                          "final_price": final_price, "cancellation_fee": cancellation_fee})
            total+=final_price
        for trek in self.completed_treks:
            final_price=self.get_final_price(trek)
            treks.append({"trek_id": trek.trek_id, "name": trek.name, "state": "completed",
                          "final_price": final_price, "cancellation_fee": 0})
            total+=final_price
        return {"traveler_id": self.traveler_id, "name": self.name, "membership_type": self.membership_type,
                "discount": Traveler.MEMBERSHIP_DISCOUNTS[self.membership_type], "treks": treks, "total": round(total, 2)}

<div dir="rtl">

### 2.3 `Guide`, מדריך
מחזיק את הטיולים שהמדריך משובץ אליהם ובודק את הזמינות שלו: חפיפת תאריכים ומגבלה של טיולים בחודש.
`analyze_guiding_statistics` מחזירה דוח עם טיולים שהושלמו, טיולים עתידיים, תפוסה ממוצעת בטיולים שהושלמו ופילוח לפי אזורים, בנוסף לדרגת ההסמכה ולשנות הניסיון.

</div>

In [4]:
class Guide:
    # Creates a valid guide with no treks; an invalid value raises ValueError before any field is set
    def __init__(self, guide_id, name, specialties, certification_level, experience_years, max_monthly_treks):
        if type(guide_id) is not int or guide_id<=0:
            raise ValueError(f"Invalid guide id: {guide_id}")
        if type(name) is not str or not name.strip():
            raise ValueError(f"Invalid name: {name}")
        if type(specialties) is not list or len(specialties)==0:
            raise ValueError(f"Invalid specialties: {specialties}")
        for region in specialties:
            if region not in Trek.VALID_REGIONS:
                raise ValueError(f"Invalid region: {region}")
        # A region listed twice would count twice in the guide's specialties
        if len(set(specialties))!=len(specialties):
            raise ValueError(f"Duplicate region in specialties: {specialties}")
        # Assumption: certification uses a 1 to 10 scale
        if type(certification_level) is not int or not 1<=certification_level<=10:
            raise ValueError(f"Invalid certification level: {certification_level}")
        # A whole number is also accepted, as with a trek's price, and is kept as a float below
        if type(experience_years) not in (int, float) or not math.isfinite(experience_years) or experience_years<0:
            raise ValueError(f"Invalid experience years: {experience_years}")
        if type(max_monthly_treks) is not int or max_monthly_treks<1:
            raise ValueError(f"Invalid monthly trek limit: {max_monthly_treks}")

        # Unique identifier of the guide
        self.guide_id=guide_id
        # Full name of the guide
        self.name=name
        # Regions the guide specializes in; a copy, so the calling code's list cannot change it later
        self.specialties=list(specialties)
        # Certification on a scale of 1 to 10, shown in the statistics report only
        self.certification_level=certification_level
        # Treks assigned to the guide, changed only through Trek.assign_guide and remove_trek
        self.assigned_treks=[]
        # Years of experience, can be partial such as 2.5
        self.experience_years=float(experience_years)
        # Most treks the guide can lead in one month, counted by departure month
        self.max_monthly_treks=max_monthly_treks

    # Asks to be assigned to a trek; Trek makes all the checks, so there is one place that assigns
    def assign_trek(self, trek):
        if not isinstance(trek, Trek):
            raise ValueError(f"Invalid trek: {trek}")
        trek.assign_guide(self)

    # Removes a trek from the guide on both sides; a completed trek stays, so the statistics remain correct (assumption)
    def remove_trek(self, trek_id):
        for trek in self.assigned_treks:
            if trek.trek_id==trek_id:
                if trek.status=="completed":
                    raise ValueError(f"Cannot remove completed trek {trek_id}")
                self.assigned_treks.remove(trek)
                trek.guide=None
                return
        raise ValueError(f"Trek {trek_id} is not assigned to guide {self.guide_id}")

    # Returns True if the date range overlaps no assigned trek; ignore_trek_id lets a trek skip itself when it moves
    def is_available(self, start_date, duration_days, ignore_trek_id=None):
        Trek.check_dates(start_date, duration_days)
        for trek in self.assigned_treks:
            if trek.trek_id!=ignore_trek_id and trek.overlaps(start_date, duration_days):
                return False
        return True

    # Returns a statistics report calculated on every call; a cancelled trek has already left assigned_treks, so it is never counted
    def analyze_guiding_statistics(self):
        completed=0
        occupancy_sum=0
        regions={}
        for trek in self.assigned_treks:
            regions[trek.region]=regions.get(trek.region, 0)+1
            if trek.status=="completed":
                completed+=1
                occupancy_sum+=len(trek.participants)/trek.max_participants
        # A guide with no completed treks gets 0 instead of a division by zero
        average_occupancy=0
        if completed>0:
            average_occupancy=round(occupancy_sum/completed, 2)
        return {"guide_id": self.guide_id, "name": self.name, "certification_level": self.certification_level,
                "experience_years": self.experience_years, "completed_treks": completed,
                "upcoming_treks": len(self.assigned_treks)-completed,
                "average_occupancy": average_occupancy, "regions": regions}

<div dir="rtl">

### 2.4 `Agency`, סוכנות
מחזיקה את מאגרי המטיילים, המדריכים והטיולים, ובודקת שאין מזהים כפולים.
- `generate_season_schedule_report` מקבצת את הטיולים לפי עונה ומתריעה על טיולים חופפים באותו אזור.
- `search_treks` מחפשת לפי שם, אזור, שם מדריך או דרגת קושי, בלי תלות באותיות גדולות וקטנות.
- `generate_membership_report` מסכמת את סוגי המנויים, את המטיילים הפעילים ואת ההכנסה הצפויה.

</div>

In [5]:
class Agency:
    # Creates an agency with empty repositories; an invalid value raises ValueError before any field is set
    def __init__(self, name):
        if type(name) is not str or not name.strip():
            raise ValueError(f"Invalid name: {name}")

        # Name of the agency
        self.name=name
        # Traveler repository, changed only through add_traveler
        self.travelers=[]
        # Guide repository, changed only through add_guide
        self.guides=[]
        # Trek repository, changed only through add_trek
        self.treks=[]

    # Adds a traveler to the repository, after checking the type and that the id is new
    def add_traveler(self, traveler):
        if not isinstance(traveler, Traveler):
            raise ValueError(f"Invalid traveler: {traveler}")
        for existing in self.travelers:
            if existing.traveler_id==traveler.traveler_id:
                raise ValueError(f"Traveler id {traveler.traveler_id} already exists")
        self.travelers.append(traveler)

    # Adds a guide to the repository, after checking the type and that the id is new
    def add_guide(self, guide):
        if not isinstance(guide, Guide):
            raise ValueError(f"Invalid guide: {guide}")
        for existing in self.guides:
            if existing.guide_id==guide.guide_id:
                raise ValueError(f"Guide id {guide.guide_id} already exists")
        self.guides.append(guide)

    # Adds a trek to the repository, after checking the type, that the id is new and that its guide is known to the agency
    def add_trek(self, trek):
        if not isinstance(trek, Trek):
            raise ValueError(f"Invalid trek: {trek}")
        for existing in self.treks:
            if existing.trek_id==trek.trek_id:
                raise ValueError(f"Trek id {trek.trek_id} already exists")
        if trek.guide is not None and trek.guide not in self.guides:
            raise ValueError(f"Guide {trek.guide.guide_id} of trek {trek.trek_id} is not in the agency")
        self.treks.append(trek)

    # Returns the season of a date by its month (assumption); a winter runs across two years, so December joins the January after it
    def get_season(self, start_date):
        if type(start_date) is not date:
            raise ValueError(f"Invalid date: {start_date}")
        month=start_date.month
        year=start_date.year
        if month==12:
            return f"winter {year}-{year+1}"
        if month<=2:
            return f"winter {year-1}-{year}"
        if month<=5:
            return f"spring {year}"
        if month<=8:
            return f"summer {year}"
        return f"autumn {year}"

    # Returns the reasons two treks that share a day deserve a warning; the region is the case the system allows, the others are blocked earlier and are kept as a safeguard
    def get_overlap_reasons(self, first, second):
        reasons=[]
        if first.region==second.region:
            reasons.append(f"same region: {first.region}")
        if first.guide is not None and first.guide is second.guide:
            reasons.append(f"same guide: {first.guide.name}")
        for traveler in first.participants:
            if traveler in second.participants:
                reasons.append(f"same traveler: {traveler.name}")
        return reasons

    # Returns the treks grouped by departure season and date, with overlap warnings between active treks; a cancelled trek is left out and a completed one gets no warning
    def generate_season_schedule_report(self):
        treks=[]
        for trek in self.treks:
            if trek.status!="cancelled":
                treks.append(trek)
        treks.sort(key=lambda trek: trek.start_date)

        seasons={}
        for trek in treks:
            guide_name=None
            if trek.guide is not None:
                guide_name=trek.guide.name
            row={"trek_id": trek.trek_id, "name": trek.name, "region": trek.region, "start_date": trek.start_date,
                 "end_date": trek.get_end_date(), "guide": guide_name, "status": trek.status,
                 "participants": len(trek.participants)}
            seasons.setdefault(self.get_season(trek.start_date), []).append(row)

        warnings=[]
        for i in range(len(treks)):
            for second in treks[i+1:]:
                first=treks[i]
                if first.is_active() and second.is_active() and first.overlaps(second.start_date, second.duration_days):
                    reasons=self.get_overlap_reasons(first, second)
                    if reasons:
                        warnings.append({"trek_ids": (first.trek_id, second.trek_id), "reasons": reasons})
        return {"agency": self.name, "seasons": seasons, "warnings": warnings}

    # Returns the matching treks: text searches name, region and guide name, a whole number searches the difficulty level
    def search_treks(self, search_term):
        if type(search_term) is int:
            return [trek for trek in self.treks if trek.difficulty_level==search_term]
        if type(search_term) is not str or not search_term.strip():
            raise ValueError(f"Invalid search term: {search_term}")

        term=search_term.strip().lower()
        results=[]
        for trek in self.treks:
            # A trek with no guide is searched only by its name and region
            texts=[trek.name, trek.region]
            if trek.guide is not None:
                texts.append(trek.guide.name)
            for text in texts:
                if term in text.lower():
                    results.append(trek)
                    break
        return results

    # Returns the travelers, active travelers and expected income of each membership type; the income is what current treks will pay at the final price
    def generate_membership_report(self):
        memberships={}
        for membership_type in Traveler.VALID_MEMBERSHIPS:
            memberships[membership_type]={"travelers": 0, "active_travelers": 0, "expected_income": 0}

        for traveler in self.travelers:
            row=memberships[traveler.membership_type]
            row["travelers"]+=1
            if len(traveler.current_treks)>0:
                row["active_travelers"]+=1
            for trek in traveler.current_treks:
                row["expected_income"]+=traveler.get_final_price(trek)

        total_income=0
        active_travelers=0
        for row in memberships.values():
            row["expected_income"]=round(row["expected_income"], 2)
            total_income+=row["expected_income"]
            active_travelers+=row["active_travelers"]
        return {"agency": self.name, "memberships": memberships, "total_travelers": len(self.travelers),
                "active_travelers": active_travelers, "expected_income": round(total_income, 2)}

<div dir="rtl">

### 2.5 הרחבה: `Equipment`, ציוד
פריט מלאי שהסוכנות משאילה לטיולים, למשל אוהלים או שקי שינה. הזמינות נבדקת לכל יום בנפרד, כך שאותו מלאי יכול לשרת טיולים שלא חופפים.
**שדות:** `equipment_id`, `name`, `total_quantity`, `loans`.
**מתודות:** `loan_to_trek`, `return_from_trek`, `get_available_quantity`, `can_supply`, `find_conflicts`.
אי אפשר להשאיל יותר מהמלאי הפנוי, ואי אפשר להשאיל לטיול שבוטל או הושלם.

</div>

In [6]:
class Equipment:
    # Creates a valid equipment item with no loans; an invalid value raises ValueError before any field is set
    def __init__(self, equipment_id, name, total_quantity):
        if type(equipment_id) is not int or equipment_id<=0:
            raise ValueError(f"Invalid equipment id: {equipment_id}")
        if type(name) is not str or not name.strip():
            raise ValueError(f"Invalid name: {name}")
        if type(total_quantity) is not int or total_quantity<1:
            raise ValueError(f"Invalid total quantity: {total_quantity}")

        # Unique identifier of the item
        self.equipment_id=equipment_id
        # Display name of the item, such as a tent
        self.name=name
        # How many units the agency owns
        self.total_quantity=total_quantity
        # Loans as dictionaries of trek and quantity, changed only through loan_to_trek and return_from_trek
        self.loans=[]

    # Returns the quantity out on one day; reads the live trek, because a trek cannot notify this class when it is cancelled or moved
    def get_quantity_on_loan(self, day, ignore_trek_id=None):
        quantity=0
        for loan in self.loans:
            trek=loan["trek"]
            if trek.is_active() and trek.trek_id!=ignore_trek_id and trek.overlaps(day, 1):
                quantity+=loan["quantity"]
        return quantity

    # Returns the units free for the whole range, by its busiest day; ignore_trek_id lets the calling code check a move before set_dates
    def get_available_quantity(self, start_date, duration_days, ignore_trek_id=None):
        Trek.check_dates(start_date, duration_days)
        busiest=0
        for offset in range(duration_days):
            day=start_date+timedelta(days=offset)
            busiest=max(busiest, self.get_quantity_on_loan(day, ignore_trek_id))
        # A move with set_dates can leave more on loan than the stock, which find_conflicts reports
        return max(self.total_quantity-busiest, 0)

    # Lends a quantity to a trek, only after every check has passed
    def loan_to_trek(self, trek, quantity):
        if not isinstance(trek, Trek):
            raise ValueError(f"Invalid trek: {trek}")
        if not trek.is_active():
            raise ValueError(f"Cannot lend to a {trek.status} trek")
        if type(quantity) is not int or quantity<1:
            raise ValueError(f"Invalid quantity: {quantity}")
        for loan in self.loans:
            if loan["trek"] is trek:
                raise ValueError(f"Trek {trek.trek_id} already has a loan of {self.name}")
        if quantity>self.get_available_quantity(trek.start_date, trek.duration_days):
            raise ValueError(f"Not enough {self.name} on the dates of trek {trek.trek_id}")

        self.loans.append({"trek": trek, "quantity": quantity})

    # Removes the loan of a trek and returns the quantity that came back
    def return_from_trek(self, trek_id):
        for loan in self.loans:
            if loan["trek"].trek_id==trek_id:
                self.loans.remove(loan)
                return loan["quantity"]
        raise ValueError(f"Trek {trek_id} has no loan of {self.name}")

    # Returns True if the item is on loan to the trek
    def is_loaned_to(self, trek_id):
        for loan in self.loans:
            if loan["trek"].trek_id==trek_id:
                return True
        return False

    # Returns True if the trek's loan still fits on the given dates, not counting the loan itself; set_dates uses it before moving a trek
    def can_supply(self, trek_id, start_date, duration_days):
        for loan in self.loans:
            if loan["trek"].trek_id==trek_id:
                return loan["quantity"]<=self.get_available_quantity(start_date, duration_days, trek_id)
        raise ValueError(f"Trek {trek_id} has no loan of {self.name}")

    # Returns the ids of active treks with a day over the stock, which can only happen after set_dates without this item in its list
    def find_conflicts(self):
        conflicts=[]
        for loan in self.loans:
            trek=loan["trek"]
            if trek.is_active():
                for offset in range(trek.duration_days):
                    day=trek.start_date+timedelta(days=offset)
                    if self.get_quantity_on_loan(day)>self.total_quantity:
                        conflicts.append(trek.trek_id)
                        break
        return conflicts

<div dir="rtl">

### 2.6 הרחבה: `Transport`, הסעה
רכב שמשובץ לטיולים, עם נקודת איסוף לכל טיול.
**שדות:** `transport_id`, `vehicle_type`, `seats`, `assignments`.
**מתודות:** `assign_to_trek`, `remove_from_trek`, `is_available`, `find_conflicts`, והמתודה הסטטית `has_enough_seats`.
אותו רכב לא משובץ לשני טיולים חופפים, ו-`has_enough_seats` בודקת שהרכבים של טיול מספיקים ל-`max_participants` ועוד המדריך.
`Trek.set_dates` מקבלת רשימות אופציונליות של רכבים וציוד, כדי לבדוק שגם הם פנויים לפני שמזיזים טיול.

</div>

In [7]:
class Transport:
    # Creates a valid vehicle with no assignments; an invalid value raises ValueError before any field is set
    def __init__(self, transport_id, vehicle_type, seats):
        if type(transport_id) is not int or transport_id<=0:
            raise ValueError(f"Invalid transport id: {transport_id}")
        if type(vehicle_type) is not str or not vehicle_type.strip():
            raise ValueError(f"Invalid vehicle type: {vehicle_type}")
        # Assumption: the smallest useful vehicle holds a guide and one participant
        if type(seats) is not int or seats<2:
            raise ValueError(f"Invalid seats: {seats}")

        # Unique identifier of the vehicle
        self.transport_id=transport_id
        # Kind of vehicle, such as a minibus
        self.vehicle_type=vehicle_type
        # Number of seats, including the guide's
        self.seats=seats
        # Assignments as dictionaries of trek and pickup point, changed only through assign_to_trek and remove_from_trek
        self.assignments=[]

    # Returns True if the range overlaps no active trek; reads the live trek, because a trek cannot notify this class when it is cancelled or moved
    def is_available(self, start_date, duration_days, ignore_trek_id=None):
        Trek.check_dates(start_date, duration_days)
        for assignment in self.assignments:
            trek=assignment["trek"]
            if trek.is_active() and trek.trek_id!=ignore_trek_id and trek.overlaps(start_date, duration_days):
                return False
        return True

    # Assigns the vehicle to a trek with its pickup point, only after every check has passed
    def assign_to_trek(self, trek, pickup_point):
        if not isinstance(trek, Trek):
            raise ValueError(f"Invalid trek: {trek}")
        if not trek.is_active():
            raise ValueError(f"Cannot assign a vehicle to a {trek.status} trek")
        if type(pickup_point) is not str or not pickup_point.strip():
            raise ValueError(f"Invalid pickup point: {pickup_point}")
        for assignment in self.assignments:
            if assignment["trek"] is trek:
                raise ValueError(f"Vehicle {self.transport_id} is already assigned to trek {trek.trek_id}")
        if not self.is_available(trek.start_date, trek.duration_days):
            raise ValueError(f"Vehicle {self.transport_id} is not available on these dates")

        self.assignments.append({"trek": trek, "pickup_point": pickup_point})

    # Removes the assignment of a trek
    def remove_from_trek(self, trek_id):
        for assignment in self.assignments:
            if assignment["trek"].trek_id==trek_id:
                self.assignments.remove(assignment)
                return
        raise ValueError(f"Trek {trek_id} is not assigned to vehicle {self.transport_id}")

    # Returns True if the vehicle is assigned to the trek
    def is_assigned_to(self, trek_id):
        for assignment in self.assignments:
            if assignment["trek"].trek_id==trek_id:
                return True
        return False

    # Returns True if the trek's vehicles together seat everyone; static, because one vehicle does not know the others
    @staticmethod
    def has_enough_seats(trek, vehicles):
        if not isinstance(trek, Trek):
            raise ValueError(f"Invalid trek: {trek}")
        if type(vehicles) is not list:
            raise ValueError(f"Invalid vehicles: {vehicles}")
        total_seats=0
        counted=[]
        for vehicle in vehicles:
            if not isinstance(vehicle, Transport):
                raise ValueError(f"Invalid vehicle: {vehicle}")
            # A vehicle listed twice or serving another trek would inflate the total
            if vehicle in counted:
                raise ValueError(f"Vehicle {vehicle.transport_id} is listed twice")
            if not vehicle.is_assigned_to(trek.trek_id):
                raise ValueError(f"Vehicle {vehicle.transport_id} is not assigned to trek {trek.trek_id}")
            counted.append(vehicle)
            total_seats+=vehicle.seats
        # Assumption: one seat stays for the guide, and the trek can reach max_participants
        return total_seats>=trek.max_participants+1

    # Returns pairs of ids of overlapping active treks, which can only happen after set_dates without this vehicle in its list
    def find_conflicts(self):
        active=[]
        for assignment in self.assignments:
            if assignment["trek"].is_active():
                active.append(assignment["trek"])

        conflicts=[]
        for i in range(len(active)):
            for second in active[i+1:]:
                if active[i].overlaps(second.start_date, second.duration_days):
                    conflicts.append((active[i].trek_id, second.trek_id))
        return conflicts

<div dir="rtl">

## 3. הדגמה מהירה
טיול אחד עם מדריך, מטייל, ציוד ורכב, לפני הבדיקות המפורטות.

</div>

In [8]:
start = date.today() + timedelta(days=45)

agency = Agency("Field Treks")
guide = Guide(1, "Noa", ["Negev"], 7, 4.5, 3)
traveler = Traveler(1, "Dana", 30, "annual", 6)
trek = Trek(1, "Ramon Crater", "Negev", 5, 1, 12, start, 3, 1000)

agency.add_guide(guide)
agency.add_traveler(traveler)
guide.assign_trek(trek)
agency.add_trek(trek)
traveler.register_to_trek(trek)

tents = Equipment(1, "Tent", 20)
tents.loan_to_trek(trek, 6)
bus = Transport(1, "Minibus", 13)
bus.assign_to_trek(trek, "Beer Sheva")

print("status:", trek.status)                                    # confirmed
print("final price:", traveler.get_final_price(trek))            # 900.0, annual membership
print("enough seats:", Transport.has_enough_seats(trek, [bus]))  # True
print("search 'negev':", agency.search_treks("negev")[0].name)   # Ramon Crater

# A broken rule raises ValueError, and the calling code decides what to do
try:
    Traveler(2, "Too Old", 130, "basic", 3)
except ValueError as error:
    print("rejected:", error)

status: confirmed
final price: 900.0
enough seats: True
search 'negev': Ramon Crater
rejected: Invalid age: 130


<div dir="rtl">

## 4. בדיקות והדגמות לכל מחלקה ומתודה
כל בדיקה מדפיסה `OK` כשפעולה תקינה הצליחה, `REJECTED` עם הסיבה כשפעולה אסורה נדחתה כמצופה, ו-`UNEXPECTED` כשהתוצאה שונה מהצפוי. בסוף מופיע סיכום. כל התאריכים יחסיים להיום, כך שהתוצאות זהות בכל יום הרצה.

התא הראשון מגדיר פונקציות עזר להרצת הבדיקות ולספירת התוצאות.

</div>

In [9]:
# Demonstrations of every class and method: a valid use, edge cases and rejected actions.
# Every date is relative to the real date.today(), so the file gives the same results on any day it runs.
# To show the past, a trek's dates are moved with set_dates; "today" itself never changes.

TODAY=date.today()
# The first day of the month two months from now, for checks that need two treks in one month
MONTH_START=(TODAY.replace(day=1)+timedelta(days=62)).replace(day=1)

# Counts the results, so the summary at the end shows whether anything behaved unexpectedly
results={"passed": 0, "failed": 0}


# Returns the date the given number of days from today, negative for the past
def days_from_today(days):
    return TODAY+timedelta(days=days)


# Prints a section title
def section(title):
    print(f"\n{'='*70}\n{title}\n{'='*70}")


# Runs an action that should succeed and prints what it returned
# The action is passed as a lambda, so the call runs only inside the try block
def expect_ok(description, action):
    try:
        result=action()
        if result is None:
            print(f"  OK        {description}")
        else:
            print(f"  OK        {description} -> {result}")
        results["passed"]+=1
        return result
    except ValueError as error:
        print(f"  FAILED    {description}: rejected unexpectedly ({error})")
        results["failed"]+=1


# Runs an action that should be rejected and prints the reason the class gave
def expect_rejected(description, action):
    try:
        action()
        print(f"  FAILED    {description}: accepted, but a rejection was expected")
        results["failed"]+=1
    except ValueError as error:
        print(f"  REJECTED  {description}: {error}")
        results["passed"]+=1


# Prints whether a value matches what the demonstration expects
def expect_equal(description, actual, expected):
    if actual==expected:
        print(f"  OK        {description} -> {actual}")
        results["passed"]+=1
    else:
        print(f"  FAILED    {description}: got {actual}, expected {expected}")
        results["failed"]+=1


# Short helpers that build valid objects, so each demonstration changes only what it checks
def make_trek(trek_id, days_ahead, region="Negev", difficulty=3, min_p=2, max_p=10, duration=3, price=1000):
    return Trek(trek_id, f"Trek {trek_id}", region, difficulty, min_p, max_p,
                days_from_today(days_ahead), duration, price)


def make_traveler(traveler_id, membership="basic", experience=5):
    return Traveler(traveler_id, f"Traveler {traveler_id}", 30, membership, experience)


def make_guide(guide_id, specialties=None, max_monthly=4):
    if specialties is None:
        specialties=["Negev"]
    return Guide(guide_id, f"Guide {guide_id}", specialties, 5, 8, max_monthly)

<div dir="rtl">

### 1. Constructors: valid objects and invalid input

</div>

In [10]:
section("1. Constructors: valid objects and invalid input")

print("\nTrek")
trek=expect_ok("valid trek", lambda: make_trek(1, 40))
print(f"            new trek starts empty: status={trek.status}, guide={trek.guide}, "
      f"participants={trek.participants}, waiting_list={trek.waiting_list}")
expect_ok("trek with a past date is allowed; registration to it is not", lambda: make_trek(2, -30))
expect_rejected("id 0", lambda: Trek(0, "A", "Negev", 3, 1, 5, days_from_today(10), 2, 100))
expect_rejected("id as text", lambda: Trek("1", "A", "Negev", 3, 1, 5, days_from_today(10), 2, 100))
expect_rejected("empty name", lambda: Trek(1, "   ", "Negev", 3, 1, 5, days_from_today(10), 2, 100))
expect_rejected("unknown region", lambda: Trek(1, "A", "Alps", 3, 1, 5, days_from_today(10), 2, 100))
expect_rejected("difficulty 11", lambda: Trek(1, "A", "Negev", 11, 1, 5, days_from_today(10), 2, 100))
expect_rejected("difficulty 2.5", lambda: Trek(1, "A", "Negev", 2.5, 1, 5, days_from_today(10), 2, 100))
expect_rejected("minimum above maximum", lambda: Trek(1, "A", "Negev", 3, 6, 5, days_from_today(10), 2, 100))
expect_rejected("minimum 0", lambda: Trek(1, "A", "Negev", 3, 0, 5, days_from_today(10), 2, 100))
expect_rejected("date as text", lambda: Trek(1, "A", "Negev", 3, 1, 5, "2027-01-01", 2, 100))
expect_rejected("datetime instead of date", lambda: Trek(1, "A", "Negev", 3, 1, 5, datetime.now(), 2, 100))
expect_rejected("duration 0", lambda: Trek(1, "A", "Negev", 3, 1, 5, days_from_today(10), 0, 100))
expect_rejected("negative price", lambda: Trek(1, "A", "Negev", 3, 1, 5, days_from_today(10), 2, -100))
expect_rejected("price True", lambda: Trek(1, "A", "Negev", 3, 1, 5, days_from_today(10), 2, True))
expect_rejected("infinite price", lambda: Trek(1, "A", "Negev", 3, 1, 5, days_from_today(10), 2, float("inf")))

print("\nTraveler")
traveler=expect_ok("valid traveler", lambda: Traveler(1, "Dana Levi", 28, "annual", 6))
print(f"            new traveler starts empty: current_treks={traveler.current_treks}, "
      f"completed_treks={traveler.completed_treks}")
expect_rejected("negative id", lambda: Traveler(-1, "Dana", 28, "basic", 5))
expect_rejected("empty name", lambda: Traveler(1, "", 28, "basic", 5))
expect_rejected("age 0", lambda: Traveler(1, "Dana", 0, "basic", 5))
expect_rejected("age as text", lambda: Traveler(1, "Dana", "28", "basic", 5))
expect_rejected("age 121, above the highest realistic age", lambda: Traveler(1, "Dana", 121, "basic", 5))
expect_ok("age 120 (edge)", lambda: Traveler(1, "Dana", 120, "basic", 5).age)
expect_rejected("unknown membership", lambda: Traveler(1, "Dana", 28, "gold", 5))
expect_rejected("experience 0", lambda: Traveler(1, "Dana", 28, "basic", 0))

print("\nGuide")
regions=["Negev", "Arava"]
guide=expect_ok("valid guide", lambda: Guide(1, "Yossi Cohen", regions, 7, 4.5, 3))
regions.append("Carmel")
expect_equal("the guide keeps its own copy of the specialties list", guide.specialties, ["Negev", "Arava"])
expect_rejected("empty specialties", lambda: Guide(1, "Yossi", [], 7, 4, 3))
expect_rejected("specialties as text", lambda: Guide(1, "Yossi", "Negev", 7, 4, 3))
expect_rejected("unknown region", lambda: Guide(1, "Yossi", ["Alps"], 7, 4, 3))
expect_rejected("the same region twice", lambda: Guide(1, "Yossi", ["Negev", "Negev"], 7, 4, 3))
expect_rejected("certification 11", lambda: Guide(1, "Yossi", ["Negev"], 11, 4, 3))
expect_rejected("negative experience", lambda: Guide(1, "Yossi", ["Negev"], 7, -1, 3))
expect_rejected("monthly limit 0", lambda: Guide(1, "Yossi", ["Negev"], 7, 4, 0))

print("\nAgency")
expect_ok("valid agency", lambda: Agency("Desert Steps").name)
expect_rejected("empty name", lambda: Agency(""))

print("\nEquipment")
expect_ok("valid equipment", lambda: Equipment(1, "Tent", 20).total_quantity)
expect_rejected("quantity 0", lambda: Equipment(1, "Tent", 0))
expect_rejected("empty name", lambda: Equipment(1, " ", 5))

print("\nTransport")
expect_ok("valid vehicle", lambda: Transport(1, "Minibus", 20).seats)
expect_rejected("1 seat", lambda: Transport(1, "Jeep", 1))
expect_rejected("empty vehicle type", lambda: Transport(1, "", 5))


1. Constructors: valid objects and invalid input

Trek
  OK        valid trek -> <__main__.Trek object at 0x7f6ca8227890>
            new trek starts empty: status=planned, guide=None, participants=[], waiting_list=[]
  OK        trek with a past date is allowed; registration to it is not -> <__main__.Trek object at 0x7f6ca82274d0>
  REJECTED  id 0: Invalid trek id: 0
  REJECTED  id as text: Invalid trek id: 1
  REJECTED  empty name: Invalid name:    
  REJECTED  unknown region: Invalid region: Alps
  REJECTED  difficulty 11: Invalid difficulty level: 11
  REJECTED  difficulty 2.5: Invalid difficulty level: 2.5
  REJECTED  minimum above maximum: Invalid participant limits: 6, 5
  REJECTED  minimum 0: Invalid participant limits: 0, 5
  REJECTED  date as text: Invalid start date: 2027-01-01
  REJECTED  datetime instead of date: Invalid start date: 2026-11-10 18:55:51.291954
  REJECTED  duration 0: Invalid duration: 0
  REJECTED  negative price: Invalid price: -100
  REJECTED  price True

<div dir="rtl">

### 2. Trek: guide and dates

</div>

In [11]:
section("2. Trek: guide and dates")

print("\nassign_guide / Guide.assign_trek")
trek=make_trek(10, 30, region="Negev")
guide=make_guide(10, ["Negev"])
expect_ok("assign a Negev guide to a Negev trek", lambda: trek.assign_guide(guide))
expect_equal("both sides are updated", (trek.guide is guide, trek in guide.assigned_treks), (True, True))
expect_rejected("a second guide for the same trek", lambda: trek.assign_guide(make_guide(11)))
expect_rejected("a guide from another region", lambda: make_trek(11, 60, region="Galilee").assign_guide(guide))
expect_rejected("a guide busy on an overlapping trek", lambda: guide.assign_trek(make_trek(12, 31)))
expect_rejected("an object that is not a guide", lambda: make_trek(13, 60).assign_guide("Yossi"))
expect_rejected("Guide.assign_trek with an object that is not a trek", lambda: guide.assign_trek("trek"))

limited=make_guide(12, max_monthly=1)
first=Trek(14, "Month A", "Negev", 3, 1, 5, MONTH_START, 2, 500)
second=Trek(15, "Month B", "Negev", 3, 1, 5, MONTH_START+timedelta(days=10), 2, 500)
expect_ok("guide with a monthly limit of 1 takes one trek", lambda: limited.assign_trek(first))
expect_rejected("a second trek in the same month", lambda: limited.assign_trek(second))

print("\nset_dates")
trek=make_trek(16, 30)
guide=make_guide(16)
trek.assign_guide(guide)
make_trek(17, 50).assign_guide(guide)
expect_ok("move the trek 5 days later", lambda: trek.set_dates(days_from_today(35), 4))
expect_equal("new dates are kept", (trek.start_date, trek.duration_days), (days_from_today(35), 4))
expect_rejected("move onto another trek of the same guide", lambda: trek.set_dates(days_from_today(49), 3))
expect_equal("a rejected move keeps the old dates", trek.start_date, days_from_today(35))
expect_rejected("duration as text", lambda: trek.set_dates(days_from_today(40), "3"))

busy=make_traveler(16)
moving=make_trek(18, 70, region="Galilee", min_p=1)
moving.register(busy)
make_trek(19, 90, region="Galilee", min_p=1).register(busy)
expect_rejected("move onto another trek of a participant", lambda: moving.set_dates(days_from_today(90), 2))


2. Trek: guide and dates

assign_guide / Guide.assign_trek
  OK        assign a Negev guide to a Negev trek
  OK        both sides are updated -> (True, True)
  REJECTED  a second guide for the same trek: Trek 10 already has a guide
  REJECTED  a guide from another region: Guide 10 does not specialize in Galilee
  REJECTED  a guide busy on an overlapping trek: Guide 10 is not available on these dates
  REJECTED  an object that is not a guide: Invalid guide: Yossi
  REJECTED  Guide.assign_trek with an object that is not a trek: Invalid trek: trek
  OK        guide with a monthly limit of 1 takes one trek
  REJECTED  a second trek in the same month: Guide 12 reached the monthly trek limit

set_dates
  OK        move the trek 5 days later
  OK        new dates are kept -> (datetime.date(2026, 12, 15), 4)
  REJECTED  move onto another trek of the same guide: Guide 16 is not available on these dates
  OK        a rejected move keeps the old dates -> 2026-12-15
  REJECTED  duration as text:

<div dir="rtl">

### 3. Trek: registration, capacity and waiting list

</div>

In [12]:
section("3. Trek: registration, capacity and waiting list")

trek=make_trek(20, 30, difficulty=5, min_p=2, max_p=2)
dana=make_traveler(20, experience=7)
noam=make_traveler(21, experience=5)
beginner=make_traveler(22, experience=2)

print("\nhas_capacity / is_suitable")
expect_equal("empty trek has room", trek.has_capacity(), True)
expect_equal("experience 7 suits difficulty 5", trek.is_suitable(dana), True)
expect_equal("experience 5 suits difficulty 5 (equal is enough)", trek.is_suitable(noam), True)
expect_equal("experience 2 does not suit difficulty 5", trek.is_suitable(beginner), False)

print("\nregister / Traveler.register_to_trek / update_status")
expect_ok("register the first traveler", lambda: trek.register(dana))
expect_equal("one participant is below the minimum of 2", trek.status, "planned")
expect_ok("second traveler registers from the traveler side", lambda: noam.register_to_trek(trek))
expect_equal("the minimum is reached", trek.status, "confirmed")
expect_equal("both sides are updated", (noam in trek.participants, trek in noam.current_treks), (True, True))
expect_equal("the trek is full", trek.has_capacity(), False)
expect_rejected("register to a full trek", lambda: trek.register(make_traveler(23)))
expect_rejected("register the same traveler twice", lambda: trek.register(dana))
expect_rejected("an unsuitable traveler", lambda: make_trek(21, 30, difficulty=5).register(beginner))
expect_rejected("a traveler with an overlapping trek", lambda: make_trek(22, 31).register(dana))
expect_rejected("a trek that already departed", lambda: make_trek(23, -1).register(make_traveler(24)))
expect_rejected("an object that is not a traveler", lambda: make_trek(24, 30).register("Dana"))
expect_rejected("register_to_trek with an object that is not a trek", lambda: dana.register_to_trek("trek"))

print("\nadd_to_waiting_list")
basic=make_traveler(25, "basic")
premium=make_traveler(26, "premium")
expect_ok("basic traveler waits first", lambda: trek.add_to_waiting_list(basic))
expect_ok("premium traveler waits second", lambda: trek.add_to_waiting_list(premium))
expect_rejected("wait twice", lambda: trek.add_to_waiting_list(basic))
expect_rejected("a registered traveler waits", lambda: trek.add_to_waiting_list(dana))
expect_rejected("an unsuitable traveler waits", lambda: trek.add_to_waiting_list(beginner))
expect_rejected("wait for a trek that still has room", lambda: make_trek(25, 30).add_to_waiting_list(basic))

print("\ncancel_registration / promote_from_waiting_list")
expect_ok("Dana cancels", lambda: trek.cancel_registration(dana.traveler_id))
expect_equal("premium is promoted before basic, who arrived earlier",
             [t.traveler_id for t in trek.participants], [21, 26])
expect_equal("basic is still waiting", [t.traveler_id for t in trek.waiting_list], [25])
expect_equal("Dana's own list no longer has the trek", trek in dana.current_treks, False)
expect_rejected("cancel a traveler who is not registered", lambda: trek.cancel_registration(999))
expect_ok("promote when the trek is full does nothing", lambda: trek.promote_from_waiting_list())
expect_equal("nothing changed", len(trek.participants), 2)

print("\nupdate_status back to planned")
expect_ok("Noam cancels, then basic is promoted", lambda: trek.cancel_registration(noam.traveler_id))
small=make_trek(26, 30, min_p=2, max_p=5)
solo=make_traveler(27)
small.register(solo)
small.register(make_traveler(28))
expect_equal("2 of minimum 2", small.status, "confirmed")
small.cancel_registration(28)
expect_equal("after a cancellation, 1 of minimum 2", small.status, "planned")


3. Trek: registration, capacity and waiting list

has_capacity / is_suitable
  OK        empty trek has room -> True
  OK        experience 7 suits difficulty 5 -> True
  OK        experience 5 suits difficulty 5 (equal is enough) -> True
  OK        experience 2 does not suit difficulty 5 -> False

register / Traveler.register_to_trek / update_status
  OK        register the first traveler
  OK        one participant is below the minimum of 2 -> planned
  OK        second traveler registers from the traveler side
  OK        the minimum is reached -> confirmed
  OK        both sides are updated -> (True, True)
  OK        the trek is full -> False
  REJECTED  register to a full trek: Trek 20 is full
  REJECTED  register the same traveler twice: Traveler 20 is already registered
  REJECTED  an unsuitable traveler: Traveler 22 is not suitable for this trek
  REJECTED  a traveler with an overlapping trek: Traveler 20 has a schedule conflict
  REJECTED  a trek that already departed: Trek

<div dir="rtl">

### 4. Trek: refunds, cancellation and completion

</div>

In [13]:
section("4. Trek: refunds, cancellation and completion")

print("\ncalculate_refund_rate (by real today, the trek dates set the distance)")
expect_equal("40 days before departure", make_trek(30, 40).calculate_refund_rate(), 1.0)
expect_equal("30 days before departure (edge)", make_trek(31, 30).calculate_refund_rate(), 1.0)
expect_equal("20 days before departure", make_trek(32, 20).calculate_refund_rate(), 0.75)
expect_equal("10 days before departure", make_trek(33, 10).calculate_refund_rate(), 0.5)
expect_equal("7 days before departure (edge)", make_trek(34, 7).calculate_refund_rate(), 0.5)
expect_equal("3 days before departure", make_trek(35, 3).calculate_refund_rate(), 0.0)
expect_rejected("cancel date as text", lambda: make_trek(36, 10).calculate_refund_rate("today"))

print("\nTraveler.cancel_trek returns the refund rate")
trek=make_trek(37, 20)
traveler=make_traveler(30)
trek.register(traveler)
expect_equal("cancel 20 days ahead", traveler.cancel_trek(37), 0.75)
expect_rejected("cancel a trek the traveler is not registered to", lambda: traveler.cancel_trek(37))

print("\ncancel_trek (by the agency)")
trek=make_trek(38, 30, min_p=1, max_p=1)
guide=make_guide(30)
trek.assign_guide(guide)
first=make_traveler(31)
waiting=make_traveler(32)
trek.register(first)
trek.add_to_waiting_list(waiting)
expect_ok("cancel the trek", lambda: trek.cancel_trek())
expect_equal("status", trek.status, "cancelled")
expect_equal("participants and waiting list are released", (trek.participants, trek.waiting_list), ([], []))
expect_equal("the traveler is free again", first.current_treks, [])
expect_equal("the guide is released", (trek.guide, trek in guide.assigned_treks), (None, False))
expect_rejected("cancel twice", lambda: trek.cancel_trek())
expect_rejected("register to a cancelled trek", lambda: trek.register(make_traveler(33)))
expect_rejected("move a cancelled trek", lambda: trek.set_dates(days_from_today(60), 2))

print("\ncomplete_trek / Traveler.add_completed_trek")
trek=make_trek(39, 30, min_p=2)
guide=make_guide(31)
trek.assign_guide(guide)
hikers=[make_traveler(34), make_traveler(35)]
for hiker in hikers:
    trek.register(hiker)
expect_rejected("complete before the trek ended", lambda: trek.complete_trek())
expect_rejected("add_completed_trek before the trek is completed", lambda: hikers[0].add_completed_trek(39))
# The registration happened while the trek was in the future; now its dates move to the past, so it has ended
expect_ok("move the trek to 10 days ago", lambda: trek.set_dates(days_from_today(-10), 3))
expect_ok("complete the trek", lambda: trek.complete_trek())
expect_equal("status", trek.status, "completed")
expect_equal("every participant moved it to completed_treks",
             [(trek in h.completed_treks, trek in h.current_treks) for h in hikers], [(True, False), (True, False)])
expect_rejected("complete twice", lambda: trek.complete_trek())
expect_rejected("complete a planned trek", lambda: make_trek(40, -10).complete_trek())
expect_rejected("remove a completed trek from its guide", lambda: guide.remove_trek(39))
expect_rejected("add_completed_trek for a trek the traveler never had", lambda: hikers[0].add_completed_trek(999))


4. Trek: refunds, cancellation and completion

calculate_refund_rate (by real today, the trek dates set the distance)
  OK        40 days before departure -> 1.0
  OK        30 days before departure (edge) -> 1.0
  OK        20 days before departure -> 0.75
  OK        10 days before departure -> 0.5
  OK        7 days before departure (edge) -> 0.5
  OK        3 days before departure -> 0.0
  REJECTED  cancel date as text: Invalid cancel date: today

Traveler.cancel_trek returns the refund rate
  OK        cancel 20 days ahead -> 0.75
  REJECTED  cancel a trek the traveler is not registered to: Traveler 30 is not registered to trek 37

cancel_trek (by the agency)
  OK        cancel the trek
  OK        status -> cancelled
  OK        participants and waiting list are released -> ([], [])
  OK        the traveler is free again -> []
  OK        the guide is released -> (None, False)
  REJECTED  cancel twice: Cannot cancel a cancelled trek
  REJECTED  register to a cancelled trek: Cann

<div dir="rtl">

### 5. Traveler: availability, private tour and payments

</div>

In [14]:
section("5. Traveler: availability, private tour and payments")

traveler=make_traveler(40, "premium", experience=6)
trek=make_trek(41, 30, price=1200)
trek.register(traveler)

print("\nis_available")
expect_equal("free on other dates", traveler.is_available(days_from_today(60), 3), True)
expect_equal("busy on the trek dates", traveler.is_available(days_from_today(31), 1), False)
expect_rejected("invalid duration", lambda: traveler.is_available(days_from_today(60), 0))

print("\nrequest_private_tour")
guide=make_guide(40, ["Judean Desert", "Negev"])
private=expect_ok("private tour with a free guide", lambda: traveler.request_private_tour(guide, days_from_today(50), 2))
print(f"            region={private.region} (guide's first specialty), difficulty={private.difficulty_level} "
      f"(traveler's level), price={private.price} (500 per day), status={private.status}")
expect_rejected("guide busy on those dates", lambda: make_traveler(41).request_private_tour(guide, days_from_today(51), 1))
expect_rejected("traveler busy on those dates", lambda: traveler.request_private_tour(make_guide(41), days_from_today(30), 1))
expect_rejected("a date in the past", lambda: make_traveler(42).request_private_tour(make_guide(42), days_from_today(-5), 1))
expect_rejected("an object that is not a guide", lambda: make_traveler(43).request_private_tour("Yossi", days_from_today(80), 1))

print("\ngenerate_payment_report (premium has 20% off; the fee is what a cancellation today would cost)")
pprint(traveler.generate_payment_report(), sort_dicts=False)
newcomer=make_traveler(44)
report=newcomer.generate_payment_report()
expect_equal("payment report of a traveler with no treks", (report["treks"], report["total"]), ([], 0))


5. Traveler: availability, private tour and payments

is_available
  OK        free on other dates -> True
  OK        busy on the trek dates -> False
  REJECTED  invalid duration: Invalid duration: 0

request_private_tour
  OK        private tour with a free guide -> <__main__.Trek object at 0x7f6ca81ea150>
            region=Judean Desert (guide's first specialty), difficulty=6 (traveler's level), price=1000 (500 per day), status=confirmed
  REJECTED  guide busy on those dates: Guide 40 is not available on these dates
  REJECTED  traveler busy on those dates: Traveler 40 is not available on these dates
  REJECTED  a date in the past: Cannot request a private tour that has already departed
  REJECTED  an object that is not a guide: Invalid guide: Yossi

generate_payment_report (premium has 20% off; the fee is what a cancellation today would cost)
{'traveler_id': 40,
 'name': 'Traveler 40',
 'membership_type': 'premium',
 'discount': 0.2,
 'treks': [{'trek_id': 41,
            'name':

<div dir="rtl">

### 6. Guide: availability, removal and statistics

</div>

In [15]:
section("6. Guide: availability, removal and statistics")

guide=make_guide(50, ["Negev", "Arava"])
upcoming=make_trek(50, 30)
done=make_trek(51, 60, region="Arava", max_p=4, min_p=2)
upcoming.assign_guide(guide)
done.assign_guide(guide)
done.register(make_traveler(50))
done.register(make_traveler(51))
done.set_dates(days_from_today(-20), 2)
done.complete_trek()

print("\nis_available")
expect_equal("free on other dates", guide.is_available(days_from_today(100), 2), True)
expect_equal("busy on the trek dates", guide.is_available(days_from_today(30), 2), False)

print("\nanalyze_guiding_statistics (completed treks, upcoming treks, average occupancy, regions)")
pprint(guide.analyze_guiding_statistics(), sort_dicts=False)
stats=make_guide(51).analyze_guiding_statistics()
expect_equal("statistics of a guide with no treks, with no division by zero",
             (stats["completed_treks"], stats["upcoming_treks"], stats["average_occupancy"], stats["regions"]), (0, 0, 0, {}))

print("\nremove_trek")
expect_ok("remove the upcoming trek", lambda: guide.remove_trek(50))
expect_equal("both sides are updated", (upcoming.guide, upcoming in guide.assigned_treks), (None, False))
expect_rejected("remove a trek that is not assigned", lambda: guide.remove_trek(50))


6. Guide: availability, removal and statistics

is_available
  OK        free on other dates -> True
  OK        busy on the trek dates -> False

analyze_guiding_statistics (completed treks, upcoming treks, average occupancy, regions)
{'guide_id': 50,
 'name': 'Guide 50',
 'certification_level': 5,
 'experience_years': 8.0,
 'completed_treks': 1,
 'upcoming_treks': 1,
 'average_occupancy': 0.5,
 'regions': {'Negev': 1, 'Arava': 1}}
  OK        statistics of a guide with no treks, with no division by zero -> (0, 0, 0, {})

remove_trek
  OK        remove the upcoming trek
  OK        both sides are updated -> (None, False)
  REJECTED  remove a trek that is not assigned: Trek 50 is not assigned to guide 50


<div dir="rtl">

### 7. Agency: repositories, search and reports

</div>

In [16]:
section("7. Agency: repositories, search and reports")

agency=Agency("Desert Steps")
guide=make_guide(60, ["Negev", "Galilee"])
other_guide=make_guide(61, ["Galilee"])

print("\nadd_guide / add_traveler / add_trek")
expect_ok("add a guide", lambda: agency.add_guide(guide))
expect_rejected("add a guide with the same id", lambda: agency.add_guide(make_guide(60)))
expect_rejected("add an object that is not a guide", lambda: agency.add_guide("guide"))
travelers=[make_traveler(60, "basic"), make_traveler(61, "annual"), make_traveler(62, "premium")]
for t in travelers:
    expect_ok(f"add traveler {t.traveler_id}", lambda: agency.add_traveler(t))
expect_rejected("add a traveler with the same id", lambda: agency.add_traveler(make_traveler(60)))

ramon=Trek(60, "Ramon Crater", "Negev", 4, 1, 10, days_from_today(30), 3, 1000)
nahal=Trek(61, "Nahal Amud", "Galilee", 2, 1, 10, days_from_today(31), 2, 600)
arbel=Trek(62, "Mount Arbel", "Galilee", 3, 1, 10, days_from_today(32), 1, 300)
ramon.assign_guide(guide)
nahal.assign_guide(other_guide)
expect_ok("add a trek with a guide from the agency", lambda: agency.add_trek(ramon))
expect_rejected("add a trek whose guide is not in the agency", lambda: agency.add_trek(nahal))
agency.add_guide(other_guide)
expect_ok("add it after the guide joins", lambda: agency.add_trek(nahal))
expect_ok("add a trek with no guide yet", lambda: agency.add_trek(arbel))
expect_rejected("add the same trek id twice", lambda: agency.add_trek(ramon))
ramon.register(travelers[0])
ramon.register(travelers[2])
nahal.register(travelers[1])

print("\nsearch_treks (name, region, guide name, difficulty)")
expect_equal("by name 'crater'", [t.trek_id for t in agency.search_treks("crater")], [60])
expect_equal("by region 'galilee'", [t.trek_id for t in agency.search_treks("galilee")], [61, 62])
expect_equal("by guide name 'Guide 61'", [t.trek_id for t in agency.search_treks("Guide 61")], [61])
expect_equal("by difficulty 3", [t.trek_id for t in agency.search_treks(3)], [62])
expect_equal("no match", agency.search_treks("Everest"), [])
expect_rejected("empty search term", lambda: agency.search_treks("  "))

print("\ngenerate_season_schedule_report (Nahal Amud and Mount Arbel share a day in the Galilee)")
pprint(agency.generate_season_schedule_report(), sort_dicts=False)

print("\ngenerate_membership_report")
pprint(agency.generate_membership_report(), sort_dicts=False)

print("\nreports and search on an empty agency")
empty=Agency("Empty Agency")
expect_equal("season schedule", empty.generate_season_schedule_report(), {"agency": "Empty Agency", "seasons": {}, "warnings": []})
membership=empty.generate_membership_report()
expect_equal("membership report totals", (membership["total_travelers"], membership["active_travelers"], membership["expected_income"]), (0, 0, 0))
expect_equal("search", empty.search_treks("Negev"), [])


7. Agency: repositories, search and reports

add_guide / add_traveler / add_trek
  OK        add a guide
  REJECTED  add a guide with the same id: Guide id 60 already exists
  REJECTED  add an object that is not a guide: Invalid guide: guide
  OK        add traveler 60
  OK        add traveler 61
  OK        add traveler 62
  REJECTED  add a traveler with the same id: Traveler id 60 already exists
  OK        add a trek with a guide from the agency
  REJECTED  add a trek whose guide is not in the agency: Guide 61 of trek 61 is not in the agency
  OK        add it after the guide joins
  OK        add a trek with no guide yet
  REJECTED  add the same trek id twice: Trek id 60 already exists

search_treks (name, region, guide name, difficulty)
  OK        by name 'crater' -> [60]
  OK        by region 'galilee' -> [61, 62]
  OK        by guide name 'Guide 61' -> [61]
  OK        by difficulty 3 -> [62]
  OK        no match -> []
  REJECTED  empty search term: Invalid search term:   

ge

<div dir="rtl">

### 8. Extension: Equipment

</div>

In [17]:
section("8. Extension: Equipment")

tents=Equipment(1, "Tent", 10)
trek_a=make_trek(70, 30, duration=3)
trek_b=make_trek(71, 31, duration=3)
trek_c=make_trek(72, 60, duration=2)

print("\nloan_to_trek / get_available_quantity / get_quantity_on_loan")
expect_ok("lend 6 tents to trek A", lambda: tents.loan_to_trek(trek_a, 6))
expect_equal("free on the dates of trek B, which overlap A", tents.get_available_quantity(trek_b.start_date, 3), 4)
expect_equal("on loan on the first day of A", tents.get_quantity_on_loan(trek_a.start_date), 6)
expect_rejected("lend 5 to trek B, only 4 are free", lambda: tents.loan_to_trek(trek_b, 5))
expect_ok("lend 4 to trek B", lambda: tents.loan_to_trek(trek_b, 4))
expect_ok("lend all 10 to trek C, on other dates", lambda: tents.loan_to_trek(trek_c, 10))
expect_rejected("a second loan to the same trek", lambda: tents.loan_to_trek(trek_a, 1))
expect_rejected("quantity 0", lambda: tents.loan_to_trek(make_trek(73, 90), 0))
expect_rejected("an object that is not a trek", lambda: tents.loan_to_trek("trek", 1))

print("\nis_loaned_to / can_supply / set_dates with equipment")
expect_equal("loaned to trek A", tents.is_loaned_to(70), True)
expect_equal("trek C cannot move onto the dates of A", tents.can_supply(72, trek_a.start_date, 2), False)
expect_rejected("set_dates checks the equipment list", lambda: trek_c.set_dates(trek_a.start_date, 2, equipment=[tents]))
expect_equal("trek C kept its dates", trek_c.start_date, days_from_today(60))

print("\nfind_conflicts (a move without the equipment list skips the check)")
expect_equal("no conflicts yet", tents.find_conflicts(), [])
trek_c.set_dates(trek_a.start_date, 2)
expect_equal("A and C are now over the stock", tents.find_conflicts(), [70, 71, 72])
trek_c.set_dates(days_from_today(60), 2)

print("\ncancelled trek frees its tents / return_from_trek")
trek_b.cancel_trek()
expect_equal("B's tents count as free again", tents.get_available_quantity(trek_b.start_date, 3), 4)
expect_equal("return A's tents", tents.return_from_trek(70), 6)
expect_rejected("return again", lambda: tents.return_from_trek(70))


8. Extension: Equipment

loan_to_trek / get_available_quantity / get_quantity_on_loan
  OK        lend 6 tents to trek A
  OK        free on the dates of trek B, which overlap A -> 4
  OK        on loan on the first day of A -> 6
  REJECTED  lend 5 to trek B, only 4 are free: Not enough Tent on the dates of trek 71
  OK        lend 4 to trek B
  OK        lend all 10 to trek C, on other dates
  REJECTED  a second loan to the same trek: Trek 70 already has a loan of Tent
  REJECTED  quantity 0: Invalid quantity: 0
  REJECTED  an object that is not a trek: Invalid trek: trek

is_loaned_to / can_supply / set_dates with equipment
  OK        loaned to trek A -> True
  OK        trek C cannot move onto the dates of A -> False
  REJECTED  set_dates checks the equipment list: Equipment 1 is not available on these dates
  OK        trek C kept its dates -> 2027-01-09

find_conflicts (a move without the equipment list skips the check)
  OK        no conflicts yet -> []
  OK        A and C are 

<div dir="rtl">

### 9. Extension: Transport

</div>

In [18]:
section("9. Extension: Transport")

bus=Transport(1, "Minibus", 8)
jeep=Transport(2, "Jeep", 4)
trek_a=make_trek(80, 30, max_p=10)
trek_b=make_trek(81, 31)

print("\nassign_to_trek / is_available / is_assigned_to")
expect_ok("assign the minibus to trek A", lambda: bus.assign_to_trek(trek_a, "Beer Sheva station"))
expect_equal("the minibus is assigned to A", bus.is_assigned_to(80), True)
expect_equal("the minibus is busy on A's dates", bus.is_available(trek_a.start_date, 1), False)
expect_rejected("assign it to trek B, which overlaps", lambda: bus.assign_to_trek(trek_b, "Tel Aviv"))
expect_rejected("empty pickup point", lambda: jeep.assign_to_trek(trek_a, " "))
cancelled=make_trek(82, 50)
cancelled.cancel_trek()
expect_rejected("assign to a cancelled trek", lambda: jeep.assign_to_trek(cancelled, "Eilat"))

print("\nhas_enough_seats (one seat is kept for the guide)")
expect_equal("8 seats for 10 travelers and a guide", Transport.has_enough_seats(trek_a, [bus]), False)
jeep.assign_to_trek(trek_a, "Beer Sheva station")
expect_equal("8 + 4 seats", Transport.has_enough_seats(trek_a, [bus, jeep]), True)
expect_rejected("the same vehicle twice", lambda: Transport.has_enough_seats(trek_a, [bus, bus]))
expect_rejected("a vehicle not assigned to the trek",
                lambda: Transport.has_enough_seats(trek_a, [bus, Transport(3, "Van", 6)]))

print("\nset_dates with vehicles / find_conflicts / remove_from_trek")
later=make_trek(84, 60)
bus.assign_to_trek(later, "Tel Aviv")
expect_rejected("set_dates checks the vehicle list", lambda: later.set_dates(trek_a.start_date, 2, vehicles=[bus]))
later.set_dates(trek_a.start_date, 2)
expect_equal("a move without the list shows up as a conflict", bus.find_conflicts(), [(80, 84)])
expect_ok("remove the minibus from the moved trek", lambda: bus.remove_from_trek(84))
expect_equal("no conflicts after the removal", bus.find_conflicts(), [])
expect_rejected("remove it again", lambda: bus.remove_from_trek(84))


9. Extension: Transport

assign_to_trek / is_available / is_assigned_to
  OK        assign the minibus to trek A
  OK        the minibus is assigned to A -> True
  OK        the minibus is busy on A's dates -> False
  REJECTED  assign it to trek B, which overlaps: Vehicle 1 is not available on these dates
  REJECTED  empty pickup point: Invalid pickup point:  
  REJECTED  assign to a cancelled trek: Cannot assign a vehicle to a cancelled trek

has_enough_seats (one seat is kept for the guide)
  OK        8 seats for 10 travelers and a guide -> False
  OK        8 + 4 seats -> True
  REJECTED  the same vehicle twice: Vehicle 1 is listed twice
  REJECTED  a vehicle not assigned to the trek: Vehicle 3 is not assigned to trek 80

set_dates with vehicles / find_conflicts / remove_from_trek
  REJECTED  set_dates checks the vehicle list: Vehicle 1 is not available on these dates
  OK        a move without the list shows up as a conflict -> [(80, 84)]
  OK        remove the minibus from the m

<div dir="rtl">

### Summary

</div>

In [19]:
section("Summary")
print(f"\n{results['passed']} checks behaved as expected, {results['failed']} did not")


Summary

186 checks behaved as expected, 0 did not


<div dir="rtl">

## 5. הסימולציה המלאה
המחלקה `AgencySimulation` מדמה את הסוכנות כמערכת שלמה. ל-`Agency` אין שדות לרכבים ולציוד, ולכן הסימולציה מחזיקה אותם.
- `load_sample_data` בונה את הסוכנות "Desert Steps" עם 4 מדריכים, 12 מטיילים, 7 טיולים, 2 רכבים ו-2 פריטי ציוד.
- `finish_trek` מזיזה טיול לעבר ומסיימת אותו. "היום" לא משתנה.
- כל צעד מסומן `OK`, `REJECTED` (נדחה כמצופה, עם הסיבה) או `UNEXPECTED`.

</div>

In [20]:
# The simulation of the field trek agency, and the main file of the project.
# It offers three modes: a full season scenario that runs on its own, an interactive menu
# for trying any action by hand, and ready made edge case scenarios.
# "Today" is always the real date.today(). To show the past, a trek's dates are moved; today never changes.


class AgencySimulation:
    # Creates a simulation with the sample agency loaded
    def __init__(self):
        # The agency, with its travelers, guides and treks
        self.agency=None
        # Vehicles and equipment are kept here, because Agency has no field for them
        self.vehicles=[]
        self.equipment=[]
        # Counts how the scripted steps behaved, so a scenario ends with a clear summary
        self.expected=0
        self.unexpected=0
        self.load_sample_data()

    # ------------------------------------------------------------------
    # Dates and printing
    # ------------------------------------------------------------------

    # Returns the date the given number of days from today, negative for the past
    def days_from_today(self, days):
        return date.today()+timedelta(days=days)

    # Prints a section title
    def title(self, text):
        print(f"\n{'='*72}\n{text}\n{'='*72}")

    # Prints a step title inside a scenario
    def step(self, text):
        print(f"\n--- {text} ---")

    # Runs an action that should succeed, prints the result and returns it
    # The action is passed as a lambda, so the call runs only inside the try block
    def expect_ok(self, description, action):
        try:
            result=action()
            print(f"  OK        {description}")
            self.expected+=1
            return result
        except ValueError as error:
            print(f"  UNEXPECTED {description}: rejected ({error})")
            self.unexpected+=1

    # Runs an action that should be rejected and prints the reason the class gave
    def expect_rejected(self, description, action):
        try:
            action()
            print(f"  UNEXPECTED {description}: accepted, but a rejection was expected")
            self.unexpected+=1
        except ValueError as error:
            print(f"  REJECTED  {description}: {error}")
            self.expected+=1

    # Prints whether a value matches what the scenario expects
    def expect_equal(self, description, actual, expected):
        if actual==expected:
            print(f"  OK        {description}: {actual}")
            self.expected+=1
        else:
            print(f"  UNEXPECTED {description}: got {actual}, expected {expected}")
            self.unexpected+=1

    # Prints the summary of a scenario and resets the counters
    def print_summary(self, name):
        print(f"\n{name} finished: {self.expected} steps behaved as expected, {self.unexpected} did not")
        self.expected=0
        self.unexpected=0

    # ------------------------------------------------------------------
    # Sample data
    # ------------------------------------------------------------------

    # Builds a small, realistic agency: 4 guides, 12 travelers, 7 treks, 2 vehicles and 2 equipment items
    # Nothing is registered yet; the season scenario and the menu start from here
    def load_sample_data(self):
        self.agency=Agency("Desert Steps")
        guides=[
            Guide(1, "Noa Ben-David", ["Negev", "Arava", "Eilat Mountains"], 8, 12, 3),
            Guide(2, "Amir Haddad", ["Galilee", "Upper Galilee", "Golan Heights"], 6, 5.5, 2),
            Guide(3, "Maya Katz", ["Judean Desert", "Dead Sea"], 9, 15, 3),
            Guide(4, "Eitan Shalev", ["Carmel", "Judean Mountains"], 4, 2, 2),
        ]
        for guide in guides:
            self.agency.add_guide(guide)

        travelers=[
            Traveler(1, "Dana Levi", 29, "premium", 8),
            Traveler(2, "Omer Cohen", 34, "annual", 6),
            Traveler(3, "Yael Mizrahi", 41, "basic", 4),
            Traveler(4, "Itai Peretz", 23, "basic", 7),
            Traveler(5, "Shira Avraham", 37, "premium", 5),
            Traveler(6, "Lior Friedman", 52, "annual", 3),
            Traveler(7, "Noam Biton", 19, "basic", 9),
            Traveler(8, "Tamar Golan", 45, "annual", 6),
            Traveler(9, "Avi Shapiro", 63, "premium", 2),
            Traveler(10, "Roni Azulay", 31, "basic", 5),
            Traveler(11, "Michal Dahan", 27, "annual", 7),
            Traveler(12, "Yoni Klein", 38, "basic", 4),
        ]
        for traveler in travelers:
            self.agency.add_traveler(traveler)

        # trek_id, name, region, difficulty, min, max, days from today, duration, price
        treks=[
            Trek(101, "Ramon Crater Traverse", "Negev", 6, 3, 8, self.days_from_today(45), 3, 1450),
            Trek(102, "Mount Arbel Sunrise", "Galilee", 3, 2, 4, self.days_from_today(20), 1, 320),
            Trek(103, "Judean Desert Canyons", "Judean Desert", 8, 2, 6, self.days_from_today(60), 4, 2200),
            Trek(104, "Carmel Forest Trails", "Carmel", 2, 4, 12, self.days_from_today(12), 2, 480),
            Trek(105, "Eilat Mountains Ridge", "Eilat Mountains", 5, 2, 10, self.days_from_today(90), 3, 1750),
            Trek(106, "Nahal Amud Stream", "Galilee", 2, 2, 8, self.days_from_today(20), 2, 540),
            Trek(107, "Golan Volcanoes", "Golan Heights", 4, 2, 8, self.days_from_today(30), 2, 690),
        ]
        for trek in treks:
            self.agency.add_trek(trek)

        self.vehicles=[Transport(1, "Minibus", 12), Transport(2, "Jeep", 4)]
        self.equipment=[Equipment(1, "Tent", 12), Equipment(2, "Sleeping bag", 15)]

    # ------------------------------------------------------------------
    # Finding objects by id
    # ------------------------------------------------------------------

    # Each finder raises ValueError for an unknown id, like the classes do, so the menu reports it the same way
    def find_trek(self, trek_id):
        for trek in self.agency.treks:
            if trek.trek_id==trek_id:
                return trek
        raise ValueError(f"No trek with id {trek_id}")

    def find_traveler(self, traveler_id):
        for traveler in self.agency.travelers:
            if traveler.traveler_id==traveler_id:
                return traveler
        raise ValueError(f"No traveler with id {traveler_id}")

    def find_guide(self, guide_id):
        for guide in self.agency.guides:
            if guide.guide_id==guide_id:
                return guide
        raise ValueError(f"No guide with id {guide_id}")

    def find_vehicle(self, transport_id):
        for vehicle in self.vehicles:
            if vehicle.transport_id==transport_id:
                return vehicle
        raise ValueError(f"No vehicle with id {transport_id}")

    def find_equipment(self, equipment_id):
        for item in self.equipment:
            if item.equipment_id==equipment_id:
                return item
        raise ValueError(f"No equipment with id {equipment_id}")

    # Returns the vehicles assigned to a trek, which set_dates needs to check a move
    def get_trek_vehicles(self, trek):
        return [vehicle for vehicle in self.vehicles if vehicle.is_assigned_to(trek.trek_id)]

    # Returns the equipment on loan to a trek, which set_dates needs to check a move
    def get_trek_equipment(self, trek):
        return [item for item in self.equipment if item.is_loaned_to(trek.trek_id)]

    # Moves a trek so it ended the given number of days ago and completes it; the trek's dates move, today does not
    def finish_trek(self, trek, ended_days_ago=1):
        if type(ended_days_ago) is not int or ended_days_ago<1:
            raise ValueError(f"Invalid number of days: {ended_days_ago}")
        # Checked before the move, so a trek that cannot be completed keeps its dates
        if trek.status!="confirmed":
            raise ValueError(f"Cannot complete a {trek.status} trek")
        trek.set_dates(self.days_from_today(-ended_days_ago-trek.duration_days+1), trek.duration_days,
                       self.get_trek_vehicles(trek), self.get_trek_equipment(trek))
        trek.complete_trek()

    # ------------------------------------------------------------------
    # Tables and reports
    # ------------------------------------------------------------------

    def print_treks(self):
        print(f"\n{'id':>8}  {'name':<24}{'region':<16}{'diff':>4}  {'start':<11}{'days':>4}  "
              f"{'price':>7}  {'guide':<15}{'taken':>6}  {'wait':>4}  status")
        for trek in self.agency.treks:
            guide_name="-"
            if trek.guide is not None:
                guide_name=trek.guide.name
            taken=f"{len(trek.participants)}/{trek.max_participants}"
            print(f"{trek.trek_id:>8}  {trek.name[:23]:<24}{trek.region:<16}{trek.difficulty_level:>4}  "
                  f"{str(trek.start_date):<11}{trek.duration_days:>4}  {trek.price:>7}  {guide_name:<15}"
                  f"{taken:>6}  {len(trek.waiting_list):>4}  {trek.status} (min {trek.min_participants})")

    def print_travelers(self):
        print(f"\n{'id':>4}  {'name':<16}{'age':>4}  {'membership':<11}{'level':>5}  current treks / completed treks")
        for traveler in self.agency.travelers:
            current=[trek.trek_id for trek in traveler.current_treks]
            completed=[trek.trek_id for trek in traveler.completed_treks]
            print(f"{traveler.traveler_id:>4}  {traveler.name:<16}{traveler.age:>4}  {traveler.membership_type:<11}"
                  f"{traveler.experience_level:>5}  {current} / {completed}")

    def print_guides(self):
        print(f"\n{'id':>4}  {'name':<16}{'cert':>4}  {'years':>5}  {'month max':>9}  specialties / assigned treks")
        for guide in self.agency.guides:
            assigned=[trek.trek_id for trek in guide.assigned_treks]
            print(f"{guide.guide_id:>4}  {guide.name:<16}{guide.certification_level:>4}  {guide.experience_years:>5}  "
                  f"{guide.max_monthly_treks:>9}  {', '.join(guide.specialties)} / {assigned}")

    def print_trek_details(self, trek):
        guide_name="no guide"
        if trek.guide is not None:
            guide_name=f"{trek.guide.name} (id {trek.guide.guide_id})"
        print(f"\nTrek {trek.trek_id}: {trek.name}, {trek.region}, difficulty {trek.difficulty_level}")
        print(f"  dates: {trek.start_date} to {trek.get_end_date()} ({trek.duration_days} days), price {trek.price}")
        print(f"  status: {trek.status}, participants {len(trek.participants)} of {trek.max_participants}, "
              f"minimum {trek.min_participants}, guide: {guide_name}")
        print(f"  participants: {[f'{t.traveler_id} {t.name}' for t in trek.participants]}")
        print(f"  waiting list: {[f'{t.traveler_id} {t.name} ({t.membership_type})' for t in trek.waiting_list]}")
        print(f"  refund rate if cancelled today: {trek.calculate_refund_rate():.0%}")
        vehicles=[f"{v.vehicle_type} {v.transport_id}" for v in self.get_trek_vehicles(trek)]
        items=[]
        for item in self.equipment:
            for loan in item.loans:
                if loan["trek"] is trek:
                    items.append(f"{loan['quantity']} x {item.name}")
        print(f"  vehicles: {vehicles}, equipment: {items}")

    def print_season_schedule(self):
        report=self.agency.generate_season_schedule_report()
        print(f"\nSeason schedule of {report['agency']}")
        for season, rows in report["seasons"].items():
            print(f"  {season}")
            for row in rows:
                print(f"    {row['start_date']} to {row['end_date']}  {row['trek_id']:>8} {row['name'][:23]:<24}"
                      f"{row['region']:<16} guide: {row['guide'] or '-':<15} {row['participants']} travelers, {row['status']}")
        if not report["warnings"]:
            print("  no overlap warnings")
        for warning in report["warnings"]:
            print(f"  WARNING treks {warning['trek_ids'][0]} and {warning['trek_ids'][1]} overlap: {', '.join(warning['reasons'])}")

    def print_membership_report(self):
        report=self.agency.generate_membership_report()
        print(f"\nMembership report of {report['agency']}")
        print(f"  {'type':<10}{'travelers':>10}{'active':>8}{'expected income':>17}")
        for membership_type, row in report["memberships"].items():
            print(f"  {membership_type:<10}{row['travelers']:>10}{row['active_travelers']:>8}{row['expected_income']:>17}")
        print(f"  {'total':<10}{report['total_travelers']:>10}{report['active_travelers']:>8}{report['expected_income']:>17}")

    def print_guide_statistics(self, guide):
        stats=guide.analyze_guiding_statistics()
        print(f"\nGuide {stats['guide_id']} {stats['name']}: certification {stats['certification_level']}, "
              f"{stats['experience_years']} years")
        print(f"  completed treks: {stats['completed_treks']}, upcoming treks: {stats['upcoming_treks']}, "
              f"average occupancy of completed treks: {stats['average_occupancy']:.0%}")
        print(f"  treks by region: {stats['regions']}")

    def print_payment_report(self, traveler):
        report=traveler.generate_payment_report()
        print(f"\nPayment report of {report['name']} ({report['membership_type']}, {report['discount']:.0%} discount)")
        for row in report["treks"]:
            print(f"  {row['trek_id']:>8} {row['name'][:23]:<24} {row['state']:<10} final price {row['final_price']:>8}"
                  f"   fee if cancelled today {row['cancellation_fee']}")
        print(f"  total: {report['total']}")

    def print_resources(self):
        print("\nVehicles")
        for vehicle in self.vehicles:
            assigned=[f"trek {a['trek'].trek_id} from {a['pickup_point']}" for a in vehicle.assignments]
            print(f"  {vehicle.transport_id} {vehicle.vehicle_type}, {vehicle.seats} seats: {assigned}, "
                  f"conflicts: {vehicle.find_conflicts()}")
        print("Equipment")
        for item in self.equipment:
            loans=[f"{loan['quantity']} to trek {loan['trek'].trek_id}" for loan in item.loans]
            print(f"  {item.equipment_id} {item.name}, {item.total_quantity} owned: {loans}, "
                  f"conflicts: {item.find_conflicts()}")

    # ------------------------------------------------------------------
    # Mode 1: the full season scenario
    # ------------------------------------------------------------------

    # Runs one season of the agency from opening to the final reports, with no input
    def run_season_scenario(self):
        self.load_sample_data()
        agency=self.agency
        trek=self.find_trek
        person=self.find_traveler
        guide=self.find_guide

        self.title(f"Season scenario of {agency.name}, today is {date.today()}")

        self.step("1. The agency opens")
        print(f"  {len(agency.guides)} guides, {len(agency.travelers)} travelers and {len(agency.treks)} treks")
        self.print_treks()
        self.expect_rejected("a second traveler with id 1",
                             lambda: agency.add_traveler(Traveler(1, "Dana Copy", 30, "basic", 5)))

        self.step("2. Guides are assigned")
        self.expect_ok("Noa to Ramon Crater (Negev)", lambda: guide(1).assign_trek(trek(101)))
        self.expect_ok("Noa to Eilat Mountains", lambda: guide(1).assign_trek(trek(105)))
        self.expect_ok("Amir to Mount Arbel (Galilee)", lambda: guide(2).assign_trek(trek(102)))
        self.expect_ok("Amir to Golan Volcanoes", lambda: guide(2).assign_trek(trek(107)))
        self.expect_ok("Maya to Judean Desert Canyons", lambda: guide(3).assign_trek(trek(103)))
        self.expect_ok("Eitan to Carmel Forest", lambda: guide(4).assign_trek(trek(104)))
        self.expect_rejected("Amir to Nahal Amud, on the same day as Mount Arbel",
                             lambda: guide(2).assign_trek(trek(106)))
        self.expect_rejected("Eitan to Nahal Amud, not his region",
                             lambda: trek(106).assign_guide(guide(4)))
        print("  Nahal Amud stays without a guide for now")

        self.step("3. Registrations")
        for traveler_id in (1, 2, 4, 11, 8):
            self.expect_ok(f"{person(traveler_id).name} to Ramon Crater",
                           lambda: person(traveler_id).register_to_trek(trek(101)))
        self.expect_equal("Ramon Crater reached its minimum of 3", trek(101).status, "confirmed")
        self.expect_rejected("Yael (level 4) to Ramon Crater (difficulty 6)",
                             lambda: person(3).register_to_trek(trek(101)))
        for traveler_id in (3, 5, 10, 6):
            self.expect_ok(f"{person(traveler_id).name} to Mount Arbel",
                           lambda: person(traveler_id).register_to_trek(trek(102)))
        self.expect_rejected("Noam to Mount Arbel, which is full", lambda: person(7).register_to_trek(trek(102)))
        self.expect_rejected("Roni to Nahal Amud, on the day of his Mount Arbel trek",
                             lambda: person(10).register_to_trek(trek(106)))
        for traveler_id in (7, 1):
            self.expect_ok(f"{person(traveler_id).name} to Judean Desert Canyons",
                           lambda: person(traveler_id).register_to_trek(trek(103)))
        self.expect_rejected("Omer (level 6) to Judean Desert Canyons (difficulty 8)",
                             lambda: person(2).register_to_trek(trek(103)))
        for traveler_id in (9, 6):
            self.expect_ok(f"{person(traveler_id).name} to Carmel Forest",
                           lambda: person(traveler_id).register_to_trek(trek(104)))
        for traveler_id in (2, 8, 4):
            self.expect_ok(f"{person(traveler_id).name} to Eilat Mountains",
                           lambda: person(traveler_id).register_to_trek(trek(105)))
        for traveler_id in (9, 12):
            self.expect_ok(f"{person(traveler_id).name} to Nahal Amud",
                           lambda: person(traveler_id).register_to_trek(trek(106)))
        for traveler_id in (5, 11):
            self.expect_ok(f"{person(traveler_id).name} to Golan Volcanoes",
                           lambda: person(traveler_id).register_to_trek(trek(107)))
        self.print_treks()

        self.step("4. The waiting list of the full Mount Arbel trek")
        self.expect_ok("Yoni (basic) joins the waiting list first", lambda: trek(102).add_to_waiting_list(person(12)))
        self.expect_ok("Dana (premium) joins second", lambda: trek(102).add_to_waiting_list(person(1)))
        self.expect_rejected("Avi (level 2) waits for a difficulty 3 trek",
                             lambda: trek(102).add_to_waiting_list(person(9)))

        self.step("5. Equipment and vehicles")
        tents=self.find_equipment(1)
        bags=self.find_equipment(2)
        minibus=self.find_vehicle(1)
        jeep=self.find_vehicle(2)
        self.expect_ok("5 tents to Ramon Crater", lambda: tents.loan_to_trek(trek(101), 5))
        self.expect_ok("5 sleeping bags to Ramon Crater", lambda: bags.loan_to_trek(trek(101), 5))
        self.expect_ok("3 tents to Judean Desert Canyons", lambda: tents.loan_to_trek(trek(103), 3))
        self.expect_rejected("13 tents to Eilat Mountains, only 12 are owned",
                             lambda: tents.loan_to_trek(trek(105), 13))
        self.expect_ok("the minibus to Ramon Crater", lambda: minibus.assign_to_trek(trek(101), "Beer Sheva Central Station"))
        self.expect_equal("12 seats are enough for 8 travelers and a guide",
                          Transport.has_enough_seats(trek(101), [minibus]), True)
        self.expect_ok("the jeep to Judean Desert Canyons", lambda: jeep.assign_to_trek(trek(103), "Jerusalem Malha"))
        self.expect_equal("4 seats are not enough for 6 travelers and a guide",
                          Transport.has_enough_seats(trek(103), [jeep]), False)
        self.expect_ok("the minibus joins as well", lambda: minibus.assign_to_trek(trek(103), "Jerusalem Malha"))
        self.expect_equal("4 + 12 seats", Transport.has_enough_seats(trek(103), [jeep, minibus]), True)

        self.step("6. Travelers cancel; the refund depends on the days left")
        rate=self.expect_ok("Shira cancels Mount Arbel, 20 days ahead", lambda: person(5).cancel_trek(102))
        print(f"            refund rate: {rate:.0%}")
        self.expect_equal("Dana (premium) is promoted before Yoni (basic), who waited longer",
                          [t.traveler_id for t in trek(102).participants], [3, 10, 6, 1])
        rate=self.expect_ok("Omer cancels Eilat Mountains, 90 days ahead", lambda: person(2).cancel_trek(105))
        print(f"            refund rate: {rate:.0%}")
        rate=self.expect_ok("Lior cancels Carmel Forest, 12 days ahead", lambda: person(6).cancel_trek(104))
        print(f"            refund rate: {rate:.0%}")
        self.expect_rejected("Lior cancels Carmel Forest again", lambda: person(6).cancel_trek(104))

        self.step("7. A private tour")
        private=self.expect_ok("Yael asks Maya for a 2 day private tour in 75 days",
                               lambda: person(3).request_private_tour(guide(3), self.days_from_today(75), 2))
        self.expect_ok("the agency adds the private tour", lambda: agency.add_trek(private))
        print(f"            {private.name}: {private.region}, difficulty {private.difficulty_level}, price {private.price}")
        self.expect_rejected("Roni asks Maya for a tour while she leads Judean Desert Canyons",
                             lambda: person(10).request_private_tour(guide(3), self.days_from_today(61), 1))

        self.step("8. The agency cancels Carmel Forest, which is below its minimum")
        carmel=trek(104)
        print(f"  Carmel Forest has {len(carmel.participants)} of the {carmel.min_participants} travelers it needs")
        self.expect_ok("cancel Carmel Forest", lambda: carmel.cancel_trek())
        self.expect_equal("Carmel Forest left Avi's current treks", carmel not in person(9).current_treks, True)
        self.expect_equal("Carmel Forest left Eitan's assigned treks", carmel not in guide(4).assigned_treks, True)
        self.expect_rejected("register to the cancelled trek", lambda: person(12).register_to_trek(carmel))

        self.step("9. Treks end (their dates move to the past, today stays)")
        self.expect_ok("Ramon Crater ended 12 days ago and is completed", lambda: self.finish_trek(trek(101), 12))
        self.expect_ok("Golan Volcanoes ended yesterday and is completed", lambda: self.finish_trek(trek(107), 1))
        self.expect_equal("Michal completed both", [t.trek_id for t in person(11).completed_treks], [101, 107])
        self.expect_rejected("complete Judean Desert Canyons, which has not ended", lambda: trek(103).complete_trek())
        self.expect_equal("the minibus has no conflicts after the move", minibus.find_conflicts(), [])

        self.step("10. End of season reports")
        self.print_treks()
        self.print_season_schedule()
        self.print_membership_report()
        for g in agency.guides:
            self.print_guide_statistics(g)
        self.print_payment_report(person(1))
        self.print_payment_report(person(3))
        print("\nSearch")
        for term in ("galilee", "Noa", 8):
            print(f"  '{term}': {[t.name for t in agency.search_treks(term)]}")
        self.print_resources()

        self.print_summary("Season scenario")

    # ------------------------------------------------------------------
    # Mode 3: edge case scenarios, each with its own small agency
    # ------------------------------------------------------------------

    # Builds a small trek for an edge case; the dates are days from today
    def make_trek(self, trek_id, days_ahead, region="Negev", difficulty=3, min_p=2, max_p=10, duration=2):
        return Trek(trek_id, f"Edge trek {trek_id}", region, difficulty, min_p, max_p,
                    self.days_from_today(days_ahead), duration, 800)

    def make_traveler(self, traveler_id, membership="basic", experience=5):
        return Traveler(traveler_id, f"Traveler {traveler_id}", 30, membership, experience)

    def make_guide(self, guide_id, max_monthly=4):
        return Guide(guide_id, f"Guide {guide_id}", ["Negev"], 5, 5, max_monthly)

    def edge_waiting_list(self):
        self.step("Full trek: the waiting list is promoted by membership, then by arrival")
        trek=self.make_trek(1, 30, max_p=2)
        first, second=self.make_traveler(1), self.make_traveler(2)
        trek.register(first)
        trek.register(second)
        waiting=[self.make_traveler(3, "basic"), self.make_traveler(4, "annual"),
                 self.make_traveler(5, "premium"), self.make_traveler(6, "premium")]
        for traveler in waiting:
            trek.add_to_waiting_list(traveler)
        print(f"  waiting in order: {[(t.traveler_id, t.membership_type) for t in trek.waiting_list]}")
        trek.cancel_registration(1)
        self.expect_equal("first freed place goes to the first premium", trek.participants[-1].traveler_id, 5)
        trek.cancel_registration(2)
        self.expect_equal("second goes to the next premium", trek.participants[-1].traveler_id, 6)
        trek.cancel_registration(5)
        self.expect_equal("then annual comes before the basic who waited longest", trek.participants[-1].traveler_id, 4)
        busy=self.make_traveler(7, "premium")
        self.make_trek(2, 30, min_p=1).register(busy)
        full=self.make_trek(3, 30, max_p=1, min_p=1)
        full.register(self.make_traveler(8))
        full.add_to_waiting_list(busy)
        full.add_to_waiting_list(self.make_traveler(9, "basic"))
        full.cancel_registration(8)
        self.expect_equal("a premium who became busy is skipped, the basic gets the place",
                          full.participants[0].traveler_id, 9)

    def edge_schedule_conflicts(self):
        self.step("Schedule conflicts of travelers and guides")
        guide=self.make_guide(1)
        traveler=self.make_traveler(1)
        first=self.make_trek(1, 30, duration=3)
        first.assign_guide(guide)
        first.register(traveler)
        self.expect_rejected("traveler: a trek on the last day of the first",
                             lambda: self.make_trek(2, 32).register(traveler))
        self.expect_ok("traveler: a trek the day after the first ends", lambda: self.make_trek(3, 33).register(traveler))
        self.expect_rejected("guide: a trek that starts a day before and overlaps",
                             lambda: self.make_trek(4, 29).assign_guide(guide))
        self.expect_rejected("move the first trek onto the traveler's second trek",
                             lambda: first.set_dates(self.days_from_today(33), 1))

    def edge_monthly_limit(self):
        self.step("A guide's monthly limit, counted by departure month")
        month_start=(date.today().replace(day=1)+timedelta(days=62)).replace(day=1)
        guide=self.make_guide(1, max_monthly=2)
        treks=[Trek(i, f"Month trek {i}", "Negev", 3, 1, 5, month_start+timedelta(days=7*(i-1)), 2, 500) for i in (1, 2, 3)]
        self.expect_ok("first trek of the month", lambda: guide.assign_trek(treks[0]))
        self.expect_ok("second trek of the month", lambda: guide.assign_trek(treks[1]))
        self.expect_rejected("third trek of the same month", lambda: guide.assign_trek(treks[2]))
        next_month=Trek(4, "Next month", "Negev", 3, 1, 5, month_start+timedelta(days=40), 2, 500)
        self.expect_ok("a trek in the next month", lambda: guide.assign_trek(next_month))

    def edge_refund_tiers(self):
        self.step("Refund tiers by days before departure, including the edges")
        for days, rate in ((40, 1.0), (30, 1.0), (29, 0.75), (14, 0.75), (13, 0.5), (7, 0.5), (6, 0.0), (1, 0.0)):
            self.expect_equal(f"{days} days before", self.make_trek(days, days).calculate_refund_rate(), rate)

    def edge_minimum(self):
        self.step("A trek drops below its minimum and the agency cancels it")
        trek=self.make_trek(1, 30, min_p=3)
        guide=self.make_guide(1)
        trek.assign_guide(guide)
        travelers=[self.make_traveler(i) for i in (1, 2, 3)]
        for traveler in travelers:
            trek.register(traveler)
        self.expect_equal("3 of minimum 3", trek.status, "confirmed")
        travelers[0].cancel_trek(1)
        self.expect_equal("after one cancellation, 2 of 3", trek.status, "planned")
        self.expect_rejected("complete a trek that is not confirmed", lambda: trek.complete_trek())
        self.expect_ok("the agency cancels it", lambda: trek.cancel_trek())
        self.expect_equal("everyone is released", [len(t.current_treks) for t in travelers], [0, 0, 0])
        self.expect_equal("the guide is free again", guide.assigned_treks, [])

    def edge_resources(self):
        self.step("Moving a trek can leave vehicles or equipment short")
        tents=Equipment(1, "Tent", 10)
        bus=Transport(1, "Minibus", 12)
        first=self.make_trek(1, 30)
        second=self.make_trek(2, 50)
        tents.loan_to_trek(first, 7)
        tents.loan_to_trek(second, 7)
        bus.assign_to_trek(first, "Tel Aviv")
        bus.assign_to_trek(second, "Haifa")
        self.expect_rejected("move the second trek onto the first, checking its tents",
                             lambda: second.set_dates(self.days_from_today(30), 2, equipment=[tents]))
        self.expect_rejected("the same move, checking its bus", lambda: second.set_dates(self.days_from_today(30), 2, vehicles=[bus]))
        second.set_dates(self.days_from_today(30), 2)
        print("  the move without the lists is allowed, so the clash shows up in find_conflicts:")
        self.expect_equal("tent conflicts", tents.find_conflicts(), [1, 2])
        self.expect_equal("bus conflicts", bus.find_conflicts(), [(1, 2)])
        first.cancel_trek()
        self.expect_equal("cancelling the first trek frees its tents and bus", (tents.find_conflicts(), bus.find_conflicts()), ([], []))

    def edge_closed_treks(self):
        self.step("Nothing changes a cancelled or completed trek")
        cancelled=self.make_trek(1, 30)
        cancelled.cancel_trek()
        completed=self.make_trek(2, 40, min_p=1)
        traveler=self.make_traveler(1)
        completed.register(traveler)
        completed.set_dates(self.days_from_today(-5), 2)
        completed.complete_trek()
        for name, trek in (("cancelled", cancelled), ("completed", completed)):
            self.expect_rejected(f"register to a {name} trek", lambda: trek.register(self.make_traveler(9)))
            self.expect_rejected(f"assign a guide to a {name} trek", lambda: trek.assign_guide(self.make_guide(9)))
            self.expect_rejected(f"move a {name} trek", lambda: trek.set_dates(self.days_from_today(60), 2))
            self.expect_rejected(f"cancel a {name} trek", lambda: trek.cancel_trek())
            self.expect_rejected(f"lend tents to a {name} trek", lambda: Equipment(1, "Tent", 5).loan_to_trek(trek, 1))
        self.expect_rejected("register to a trek that already departed", lambda: self.make_trek(3, 0).register(traveler))

    def edge_invalid_input(self):
        self.step("Invalid input is rejected before anything is created")
        self.expect_rejected("trek with minimum above maximum",
                             lambda: Trek(1, "A", "Negev", 3, 9, 5, self.days_from_today(10), 2, 100))
        self.expect_rejected("trek in an unknown region",
                             lambda: Trek(1, "A", "Sinai", 3, 1, 5, self.days_from_today(10), 2, 100))
        self.expect_rejected("traveler aged 130", lambda: Traveler(1, "A", 130, "basic", 5))
        self.expect_rejected("guide with the same region twice", lambda: Guide(1, "A", ["Negev", "Negev"], 5, 5, 2))
        self.expect_rejected("vehicle with one seat", lambda: Transport(1, "Bike", 1))
        self.expect_rejected("search with an empty term", lambda: Agency("A").search_treks(" "))

    # Returns the edge case scenarios in menu order, as pairs of a title and a method
    def get_edge_cases(self):
        return [("Waiting list priority", self.edge_waiting_list),
                ("Schedule conflicts", self.edge_schedule_conflicts),
                ("Guide's monthly limit", self.edge_monthly_limit),
                ("Refund tiers", self.edge_refund_tiers),
                ("Trek below its minimum", self.edge_minimum),
                ("Vehicles and equipment after a move", self.edge_resources),
                ("Cancelled and completed treks", self.edge_closed_treks),
                ("Invalid input", self.edge_invalid_input)]

    # Runs every edge case scenario and prints one summary
    def run_all_edge_cases(self):
        self.title("Edge case scenarios")
        for name, scenario in self.get_edge_cases():
            scenario()
        self.print_summary("Edge case scenarios")

    # ------------------------------------------------------------------
    # Mode 2: the interactive menu
    # ------------------------------------------------------------------

    # Input helpers ask again until the value can be read, so a typing mistake never stops the simulation
    def ask_text(self, prompt):
        while True:
            text=input(f"{prompt}: ").strip()
            if text:
                return text
            print("  please type a value")

    def ask_int(self, prompt):
        while True:
            text=input(f"{prompt}: ").strip()
            try:
                return int(text)
            except ValueError:
                print("  please type a whole number")

    def ask_number(self, prompt):
        while True:
            text=input(f"{prompt}: ").strip()
            try:
                value=float(text)
            except ValueError:
                print("  please type a number")
                continue
            # A whole number stays an int, so a price of 300 is shown as 300
            if value.is_integer():
                return int(value)
            return value

    # Accepts a date as YYYY-MM-DD, or as +N / -N days from today
    def ask_date(self, prompt):
        while True:
            text=input(f"{prompt} (YYYY-MM-DD, or +N days from today): ").strip()
            try:
                if text.startswith(("+", "-")):
                    return self.days_from_today(int(text))
                return date.fromisoformat(text)
            except ValueError:
                print("  please type a date such as 2027-03-15 or +30")

    def ask_list(self, prompt):
        return [part.strip() for part in self.ask_text(f"{prompt} (comma separated)").split(",")]

    # Each menu action is a method; the menu catches ValueError, so a rejected action prints its reason and the menu goes on
    def action_add_traveler(self):
        traveler=Traveler(self.ask_int("traveler id"), self.ask_text("full name"), self.ask_int("age"),
                          self.ask_text(f"membership {Traveler.VALID_MEMBERSHIPS}"), self.ask_int("experience 1-10"))
        self.agency.add_traveler(traveler)
        print(f"  added traveler {traveler.traveler_id}")

    def action_add_guide(self):
        guide=Guide(self.ask_int("guide id"), self.ask_text("full name"), self.ask_list("specialties"),
                    self.ask_int("certification 1-10"), self.ask_number("years of experience"),
                    self.ask_int("most treks per month"))
        self.agency.add_guide(guide)
        print(f"  added guide {guide.guide_id}")

    def action_add_trek(self):
        print(f"  regions: {', '.join(Trek.VALID_REGIONS)}")
        trek=Trek(self.ask_int("trek id"), self.ask_text("name"), self.ask_text("region"),
                  self.ask_int("difficulty 1-10"), self.ask_int("minimum participants"),
                  self.ask_int("maximum participants"), self.ask_date("start date"),
                  self.ask_int("duration in days"), self.ask_number("price"))
        self.agency.add_trek(trek)
        print(f"  added trek {trek.trek_id}")

    def action_register(self):
        traveler=self.find_traveler(self.ask_int("traveler id"))
        trek=self.find_trek(self.ask_int("trek id"))
        if not trek.has_capacity() and trek.is_active():
            answer=input("  the trek is full, join the waiting list? (y/n): ").strip().lower()
            if answer=="y":
                trek.add_to_waiting_list(traveler)
                print(f"  {traveler.name} is number {len(trek.waiting_list)} on the waiting list")
            return
        traveler.register_to_trek(trek)
        print(f"  {traveler.name} registered, the trek is {trek.status}")

    def action_cancel_registration(self):
        traveler=self.find_traveler(self.ask_int("traveler id"))
        trek_id=self.ask_int("trek id")
        rate=traveler.cancel_trek(trek_id)
        print(f"  cancelled with a refund of {rate:.0%}; the trek is {self.find_trek(trek_id).status}")

    def action_assign_guide(self):
        guide=self.find_guide(self.ask_int("guide id"))
        guide.assign_trek(self.find_trek(self.ask_int("trek id")))
        print("  guide assigned")

    def action_change_dates(self):
        trek=self.find_trek(self.ask_int("trek id"))
        trek.set_dates(self.ask_date("new start date"), self.ask_int("duration in days"),
                       self.get_trek_vehicles(trek), self.get_trek_equipment(trek))
        print(f"  the trek now runs {trek.start_date} to {trek.get_end_date()}")

    def action_cancel_trek(self):
        trek=self.find_trek(self.ask_int("trek id"))
        trek.cancel_trek()
        print("  trek cancelled, everyone was released")

    def action_finish_trek(self):
        trek=self.find_trek(self.ask_int("trek id"))
        self.finish_trek(trek, self.ask_int("how many days ago did it end (1 = yesterday)"))
        print(f"  the trek ended on {trek.get_end_date()} and is completed")

    def action_private_tour(self):
        traveler=self.find_traveler(self.ask_int("traveler id"))
        guide=self.find_guide(self.ask_int("guide id"))
        trek=traveler.request_private_tour(guide, self.ask_date("start date"), self.ask_int("duration in days"))
        self.agency.add_trek(trek)
        print(f"  private tour {trek.trek_id} created: {trek.region}, difficulty {trek.difficulty_level}, price {trek.price}")

    def action_lend_equipment(self):
        item=self.find_equipment(self.ask_int(f"equipment id {[(i.equipment_id, i.name) for i in self.equipment]}"))
        trek=self.find_trek(self.ask_int("trek id"))
        print(f"  {item.get_available_quantity(trek.start_date, trek.duration_days)} free on the trek's dates")
        item.loan_to_trek(trek, self.ask_int("quantity"))
        print("  lent")

    def action_assign_vehicle(self):
        vehicle=self.find_vehicle(self.ask_int(f"vehicle id {[(v.transport_id, v.vehicle_type) for v in self.vehicles]}"))
        trek=self.find_trek(self.ask_int("trek id"))
        vehicle.assign_to_trek(trek, self.ask_text("pickup point"))
        enough=Transport.has_enough_seats(trek, self.get_trek_vehicles(trek))
        print(f"  assigned; the trek's vehicles seat everyone: {enough}")

    def action_search(self):
        term=input("search term (text, or a whole number for difficulty): ").strip()
        if term.isdigit():
            term=int(term)
        results=self.agency.search_treks(term)
        print(f"  {len(results)} found: {[f'{t.trek_id} {t.name}' for t in results]}")

    def action_edge_cases(self):
        cases=self.get_edge_cases()
        for number, (name, scenario) in enumerate(cases, 1):
            print(f"  {number}. {name}")
        print(f"  {len(cases)+1}. Run all")
        choice=self.ask_int("edge case")
        if choice==len(cases)+1:
            self.run_all_edge_cases()
        elif 1<=choice<=len(cases):
            cases[choice-1][1]()
            self.print_summary(cases[choice-1][0])
        else:
            print("  no such edge case")

    def action_reset(self):
        self.load_sample_data()
        print("  sample data reloaded, nothing is registered")

    # Returns the menu as pairs of a label and a method, grouped by kind
    def get_menu(self):
        return [
            ("View", None),
            ("Show treks", self.print_treks),
            ("Show travelers", self.print_travelers),
            ("Show guides", self.print_guides),
            ("Trek details", lambda: self.print_trek_details(self.find_trek(self.ask_int("trek id")))),
            ("Show vehicles and equipment", self.print_resources),
            ("Create", None),
            ("Add a traveler", self.action_add_traveler),
            ("Add a guide", self.action_add_guide),
            ("Add a trek", self.action_add_trek),
            ("Registrations", None),
            ("Register a traveler (or join the waiting list)", self.action_register),
            ("Cancel a registration", self.action_cancel_registration),
            ("Request a private tour", self.action_private_tour),
            ("Trek management", None),
            ("Assign a guide", self.action_assign_guide),
            ("Change a trek's dates", self.action_change_dates),
            ("Cancel a trek (agency)", self.action_cancel_trek),
            ("Finish a trek (it moves to the past and is completed)", self.action_finish_trek),
            ("Lend equipment to a trek", self.action_lend_equipment),
            ("Assign a vehicle to a trek", self.action_assign_vehicle),
            ("Reports", None),
            ("Season schedule", self.print_season_schedule),
            ("Membership report", self.print_membership_report),
            ("Guide statistics", lambda: self.print_guide_statistics(self.find_guide(self.ask_int("guide id")))),
            ("Payment report", lambda: self.print_payment_report(self.find_traveler(self.ask_int("traveler id")))),
            ("Search treks", self.action_search),
            ("More", None),
            ("Edge case scenarios", self.action_edge_cases),
            ("Reset to the sample data", self.action_reset),
        ]

    # Shows the menu until the user chooses 0; input that ends, as from a file, also ends the menu
    def run_menu(self):
        self.title(f"Interactive menu of {self.agency.name}, today is {date.today()}")
        actions=[]
        for label, action in self.get_menu():
            if action is not None:
                actions.append((label, action))
        while True:
            print()
            number=0
            for label, action in self.get_menu():
                if action is None:
                    print(f"  [{label}]")
                else:
                    number+=1
                    print(f"  {number:>2}. {label}")
            print("   0. Back to the start menu")
            try:
                choice=input("choice: ").strip()
                if choice=="0":
                    return
                if not choice.isdigit() or not 1<=int(choice)<=len(actions):
                    print("  please choose a number from the menu")
                    continue
                label, action=actions[int(choice)-1]
                print(f"\n>> {label}")
                action()
            except ValueError as error:
                print(f"  REJECTED: {error}")
            except EOFError:
                print("\nno more input, leaving the menu")
                return


    # ------------------------------------------------------------------
    # Start menu
    # ------------------------------------------------------------------

    # Shows the start menu and runs the chosen mode, then returns to it; 0, or input that ends, exits
    def run(self):
        while True:
            print("\nField trek agency simulation")
            print("  1. season  Full season scenario, runs on its own")
            print("  2. menu    Interactive menu")
            print("  3. edge    All edge case scenarios")
            print("  0. exit")
            try:
                mode=input("choice: ").strip().lower()
            except EOFError:
                print("\nno more input, the simulation ends")
                return
            if mode in ("1", "season"):
                self.run_season_scenario()
            elif mode in ("2", "menu"):
                self.run_menu()
            elif mode in ("3", "edge"):
                self.run_all_edge_cases()
            elif mode in ("0", "exit"):
                return
            else:
                print("  please choose 1, 2, 3 or 0, or type season, menu, edge or exit")

<div dir="rtl">

### 5.1 תרחיש העונה
עשרה שלבים שרצים לבד: פתיחת הסוכנות, שיבוץ מדריכים, הרשמות, רשימת המתנה, ציוד ורכבים, ביטולים עם החזר, טיול פרטי, ביטול טיול ע"י הסוכנות, סיום טיולים ודוחות. בכל שלב מודגמים גם מקרים שבהם הרישום או השיבוץ נדחים.

</div>

In [21]:
AgencySimulation().run_season_scenario()


Season scenario of Desert Steps, today is 2026-11-10

--- 1. The agency opens ---
  4 guides, 12 travelers and 7 treks

      id  name                    region          diff  start      days    price  guide           taken  wait  status
     101  Ramon Crater Traverse   Negev              6  2026-12-25    3     1450  -                 0/8     0  planned (min 3)
     102  Mount Arbel Sunrise     Galilee            3  2026-11-30    1      320  -                 0/4     0  planned (min 2)
     103  Judean Desert Canyons   Judean Desert      8  2027-01-09    4     2200  -                 0/6     0  planned (min 2)
     104  Carmel Forest Trails    Carmel             2  2026-11-22    2      480  -                0/12     0  planned (min 4)
     105  Eilat Mountains Ridge   Eilat Mountains    5  2027-02-08    3     1750  -                0/10     0  planned (min 2)
     106  Nahal Amud Stream       Galilee            2  2026-11-30    2      540  -                 0/8     0  planned (min 2)

<div dir="rtl">

### 5.2 תרחישי קצה
כל תרחיש בונה סוכנות קטנה משלו ובודק נושא אחד: עדיפות ברשימת ההמתנה, התנגשויות בלוח הזמנים, המגבלה החודשית של מדריך, גבולות מדרגות ההחזר, טיול שלא הגיע למינימום, רכבים וציוד אחרי הזזת טיול, פעולות על טיול שבוטל או הושלם, וקלט לא תקין.

</div>

In [22]:
AgencySimulation().run_all_edge_cases()


Edge case scenarios

--- Full trek: the waiting list is promoted by membership, then by arrival ---
  waiting in order: [(3, 'basic'), (4, 'annual'), (5, 'premium'), (6, 'premium')]
  OK        first freed place goes to the first premium: 5
  OK        second goes to the next premium: 6
  OK        then annual comes before the basic who waited longest: 4
  OK        a premium who became busy is skipped, the basic gets the place: 9

--- Schedule conflicts of travelers and guides ---
  REJECTED  traveler: a trek on the last day of the first: Traveler 1 has a schedule conflict
  OK        traveler: a trek the day after the first ends
  REJECTED  guide: a trek that starts a day before and overlaps: Guide 1 is not available on these dates
  REJECTED  move the first trek onto the traveler's second trek: Traveler 1 is not available on these dates

--- A guide's monthly limit, counted by departure month ---
  OK        first trek of the month
  OK        second trek of the month
  REJECTED  t

<div dir="rtl">

### 5.3 תפריט אינטראקטיבי (אופציונלי)
התפריט מאפשר לנסות כל פעולה ידנית: יצירה, הרשמה, ביטול, שיבוץ, שינוי תאריכים ודוחות. הוא מחכה לקלט, ולכן הוא לא רץ ב-Run all. כדי להפעיל אותו, מסירים את סימן ההערה `#` ומריצים את התא.

</div>

In [23]:
# AgencySimulation().run()